# 07 — Similarity Model and Evaluation

## Purpose

This notebook develops an explainable search model for GrantScopeAI.

The model will accept a short research concept and return similar NSF and CORDIS projects based on their scientific text.

The results will support:

- the Streamlit funded-project search;
- the final project demonstration;
- comparison of proposed research ideas with previously funded work;
- transparent evaluation using the manually reviewed Day 6 reference set.

## Main modelling questions

This notebook will investigate:

1. Can simple keyword overlap retrieve relevant funded projects?
2. Does TF-IDF improve ranking quality over the keyword baseline?
3. Can cosine similarity identify scientifically related projects across NSF and CORDIS?
4. Which shared terms explain each recommendation?
5. Do manually rated strong matches appear near the top of the results?
6. Does the model generalise to research concepts outside the prepared catalysis example?

## Modelling boundaries

The model is a similarity and research-discovery tool, not a grant-acceptance predictor.

Several interpretation rules will be maintained:

- similarity does not indicate funding probability;
- high similarity does not guarantee programme eligibility;
- source, year, topic, and relevance fields will be used as filters or metadata;
- funding amount, organisation, and award year will not influence the text-similarity score;
- NSF and CORDIS funding values will remain in their native currencies;
- repeated collaborative awards may represent valid separate records but should not dominate evaluation;
- the manually reviewed benchmark is intended for qualitative model checking rather than formal accuracy measurement.

## Planned workflow

The notebook will:

1. load the integrated grant catalogue and evaluation reference set;
2. prepare a searchable text corpus;
3. build a keyword-overlap baseline;
4. create TF-IDF document vectors;
5. rank grants using cosine similarity;
6. add matched-term explanations;
7. compare model results with the manual benchmark;
8. test additional research concepts;
9. export Streamlit-ready model artifacts and result tables.

## Completion criteria

The notebook will be considered complete when:

- all modelling inputs load and validate successfully;
- the keyword baseline returns ranked results;
- the TF-IDF model produces reusable similarity scores;
- search filters work correctly;
- recommendation explanations are available;
- the manual benchmark has been evaluated;
- at least three research concepts have been tested;
- model outputs are exported, reloaded, and validated.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

# Define input folders
INTEGRATED_DATA_DIR = Path("../Data/Processed_Data/Integrated")
EDA_DATA_DIR = Path("../Data/Processed_Data/EDA")

# Define required input files
grants_path = (
    INTEGRATED_DATA_DIR
    / "grants_clean_2021_2025.csv"
)

evaluation_reference_path = (
    EDA_DATA_DIR
    / "manual_similarity_evaluation_reference.csv"
)

presentation_examples_path = (
    EDA_DATA_DIR
    / "presentation_example_grants.csv"
)

required_files = {
    "Integrated grant catalogue": grants_path,
    "Manual evaluation reference": evaluation_reference_path,
    "Presentation examples": presentation_examples_path,
}

# Confirm that all required files exist
missing_files = [
    str(path)
    for path in required_files.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(missing_files)
    )

# Load modelling inputs
grants_df = pd.read_csv(
    grants_path,
    parse_dates=["start_date", "end_date"],
    low_memory=False,
)

evaluation_reference_df = pd.read_csv(
    evaluation_reference_path,
    low_memory=False,
)

presentation_examples_df = pd.read_csv(
    presentation_examples_path,
    low_memory=False,
)

# Display the loaded dataset sizes
input_summary_df = pd.DataFrame(
    {
        "dataset": [
            "Integrated grant catalogue",
            "Manual evaluation reference",
            "Presentation examples",
        ],
        "rows": [
            len(grants_df),
            len(evaluation_reference_df),
            len(presentation_examples_df),
        ],
        "columns": [
            grants_df.shape[1],
            evaluation_reference_df.shape[1],
            presentation_examples_df.shape[1],
        ],
    }
)

input_summary_df

,dataset,rows,columns
0,Integrated grant catalogue,3339,25
1,Manual evaluation reference,8,14
2,Presentation examples,10,13


In [3]:
# Validate modelling inputs before preparing the text corpus

validation_results = [
    {
        "check": "Integrated grant row count",
        "expected": 3339,
        "actual": len(grants_df),
    },
    {
        "check": "Manual evaluation row count",
        "expected": 8,
        "actual": len(evaluation_reference_df),
    },
    {
        "check": "Presentation example row count",
        "expected": 10,
        "actual": len(presentation_examples_df),
    },
    {
        "check": "Duplicate grant catalogue keys",
        "expected": 0,
        "actual": grants_df["grant_key"].duplicated().sum(),
    },
    {
        "check": "Duplicate evaluation grant keys",
        "expected": 0,
        "actual": evaluation_reference_df[
            "grant_key"
        ].duplicated().sum(),
    },
    {
        "check": "Duplicate presentation grant keys",
        "expected": 0,
        "actual": presentation_examples_df[
            "grant_key"
        ].duplicated().sum(),
    },
    {
        "check": "Missing catalogue titles",
        "expected": 0,
        "actual": grants_df["title"].isna().sum(),
    },
    {
        "check": "Missing catalogue abstracts",
        "expected": 0,
        "actual": grants_df["abstract"].isna().sum(),
    },
]

validation_df = pd.DataFrame(validation_results)

validation_df["status"] = np.where(
    validation_df["actual"] == validation_df["expected"],
    "PASS",
    "REVIEW",
)

display(validation_df)

print(
    "Checks passed:",
    (validation_df["status"] == "PASS").sum(),
    "/",
    len(validation_df),
)

,check,expected,actual,status
0,Integrated grant row count,3339,3339,PASS
1,Manual evaluation row count,8,8,PASS
2,Presentation example row count,10,10,PASS
3,Duplicate grant catalogue keys,0,0,PASS
4,Duplicate evaluation grant keys,0,0,PASS
5,Duplicate presentation grant keys,0,0,PASS
6,Missing catalogue titles,0,0,PASS
7,Missing catalogue abstracts,0,0,PASS


Checks passed: 8 / 8


## 1. Prepare the searchable grant corpus

The similarity model will compare each user query with a combined text field built from:

- grant title;
- abstract;
- programme name.

Award amount, organisation, year, and source will remain metadata or filters and will not influence the similarity score.

Records will be checked for missing or unusually short text before modelling.

In [4]:
# Create the searchable text corpus

search_catalog_df = grants_df.copy()

text_columns = [
    "title",
    "abstract",
    "programme_name",
]

# Clean each text field while preserving the original display columns
for column in text_columns:
    search_catalog_df[f"{column}_clean"] = (
        search_catalog_df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Combine title, abstract, and programme information
search_catalog_df["search_text"] = (
    search_catalog_df["title_clean"]
    + " "
    + search_catalog_df["abstract_clean"]
    + " "
    + search_catalog_df["programme_name_clean"]
)

search_catalog_df["search_text"] = (
    search_catalog_df["search_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)

# Measure the amount of searchable text available
search_catalog_df["search_character_count"] = (
    search_catalog_df["search_text"].str.len()
)

search_catalog_df["search_word_count"] = (
    search_catalog_df["search_text"]
    .str.split()
    .str.len()
)

# Flag records that may contain too little text for reliable similarity search
minimum_word_count = 25

search_catalog_df["has_sufficient_search_text"] = (
    search_catalog_df["search_word_count"] >= minimum_word_count
)

# Create the final modelling corpus
model_catalog_df = search_catalog_df.loc[
    search_catalog_df["has_sufficient_search_text"]
].copy()

model_catalog_df = model_catalog_df.reset_index(drop=True)

corpus_summary_df = pd.DataFrame(
    {
        "metric": [
            "Original grant records",
            "Searchable grant records",
            "Records excluded for short text",
            "Minimum word count",
            "Median search words",
            "Minimum search words",
            "Maximum search words",
            "Duplicate grant keys",
            "Duplicate search texts",
        ],
        "value": [
            len(search_catalog_df),
            len(model_catalog_df),
            (
                ~search_catalog_df["has_sufficient_search_text"]
            ).sum(),
            minimum_word_count,
            model_catalog_df["search_word_count"].median(),
            model_catalog_df["search_word_count"].min(),
            model_catalog_df["search_word_count"].max(),
            model_catalog_df["grant_key"].duplicated().sum(),
            model_catalog_df["search_text"].duplicated().sum(),
        ],
    }
)

display(corpus_summary_df)

display(
    model_catalog_df[
        [
            "grant_key",
            "source",
            "title",
            "primary_topic",
            "search_word_count",
        ]
    ].head()
)

,metric,value
0,Original grant records,3339.0
1,Searchable grant records,3339.0
2,Records excluded for short text,0.0
3,Minimum word count,25.0
4,Median search words,407.0
5,Minimum search words,92.0
6,Maximum search words,1163.0
7,Duplicate grant keys,0.0
8,Duplicate search texts,396.0


,grant_key,source,title,primary_topic,search_word_count
0,CORDIS_101039636,CORDIS,"Piezoelectric Biomolecules for lead-free, Reliable, Eco-Friendly Electronics",Other AI-enabled chemistry/materials,261
1,CORDIS_101040353,CORDIS,Exploring the Molecular Properties of Atmospheric Freshly Nucleated Particles,AI-enabled chemistry,296
2,CORDIS_101040355,CORDIS,Probing (Orphan) Nuclear Receptors in Neurodegeneration,Other AI-enabled chemistry/materials,279
3,CORDIS_101040729,CORDIS,FIrst NEar-TErm ApplicationS of QUAntum Devices,AI-enabled chemistry,289
4,CORDIS_101041177,CORDIS,"Deciphering cellular and viral determinants of lytic HSV-1 infection, latency and reactivation",Other AI-enabled chemistry/materials,264


In [5]:
# Inspect grants with identical searchable text

duplicate_text_mask = model_catalog_df["search_text"].duplicated(
    keep=False
)

duplicate_text_df = model_catalog_df.loc[
    duplicate_text_mask
].copy()

# Assign one identifier to each exact-text group
duplicate_text_df["duplicate_text_group"] = (
    duplicate_text_df.groupby("search_text").ngroup() + 1
)

duplicate_group_summary_df = (
    duplicate_text_df.groupby(
        "duplicate_text_group",
        as_index=False,
    )
    .agg(
        record_count=("grant_key", "size"),
        source_count=("source", "nunique"),
        sources=("source", lambda values: ", ".join(sorted(set(values)))),
        title=("title", "first"),
        first_grant_key=("grant_key", "first"),
    )
    .sort_values(
        ["record_count", "title"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print(
    "Records in duplicate-text groups:",
    len(duplicate_text_df),
)

print(
    "Exact-text duplicate groups:",
    len(duplicate_group_summary_df),
)

print(
    "Largest duplicate group:",
    duplicate_group_summary_df["record_count"].max(),
)

display(
    duplicate_group_summary_df.head(15)
)

Records in duplicate-text groups: 680
Exact-text duplicate groups: 284
Largest duplicate group: 6


,duplicate_text_group,record_count,source_count,sources,title,first_grant_key
0,68,6,1,NSF,Collaborative Research: CyberTraining: Implementation: Medium: Establishing Sustainable Ecosystem for Computational ...,NSF_2118155
1,125,6,1,NSF,Collaborative Research: FMitF: Track I: Synthesis and Verification of In-Memory Computing Systems using Formal Methods,NSF_2319399
2,136,6,1,NSF,Collaborative Research: FW-HTF-R: Future of Construction Workplace Health Monitoring,NSF_2222619
3,153,6,1,NSF,Collaborative Research: III: Medium: New Machine Learning Empowered Nanoinformatics System for Advancing Nanomateria...,NSF_2211489
4,101,5,1,NSF,Collaborative Research: DMREF: Organic Materials Architectured for Researching Vibronic Excitations with Light in th...,NSF_2323666
5,193,5,1,NSF,Collaborative Research: PPoSS: Planning: Cross-layer Coordination and Optimization for Scalable and Sparse Tensor Ne...,NSF_2217010
6,26,4,1,NSF,CDS&E/Collaborative Research: Physics-Informed Machine Learning for Tailoring the Multidirectional Mechanical Proper...,NSF_2347658
7,34,4,1,NSF,Collaborative Research: A novel approach to study monomethylmercury in natural phytoplankton assemblages,NSF_2343142
8,58,4,1,NSF,Collaborative Research: CNS Core: Medium: Data Augmentation and Adaptive Learning for Next Generation Wireless Spect...,NSF_2107014
9,81,4,1,NSF,Collaborative Research: DMREF: AI-enabled Automated design of ultrastrong and ultraelastic metallic alloys,NSF_2323765


### Handling repeated collaborative awards

The source catalogue retains all 3,339 awards, but identical search texts will be collapsed into one modelling document.

This prevents repeated collaborative NSF records from dominating similarity rankings. A separate group mapping will preserve the related grant keys and organisations for later display.

In [6]:
# Collapse identical search texts into one modelling document
# while preserving a mapping to every underlying grant record

# Assign one group identifier to each unique search text
model_catalog_df["text_group_id"] = (
    pd.factorize(
        model_catalog_df["search_text"],
        sort=True,
    )[0]
    + 1
)

# Preserve the relationship between modelling documents
# and every original grant record
duplicate_group_members_df = (
    model_catalog_df[
        [
            "text_group_id",
            "grant_key",
            "source",
            "title",
            "organisation_name",
            "award_year",
            "source_url",
        ]
    ]
    .copy()
    .sort_values(
        ["text_group_id", "grant_key"]
    )
    .reset_index(drop=True)
)

# Create group-level metadata
text_group_metadata_df = (
    model_catalog_df.groupby(
        "text_group_id",
        as_index=False,
    )
    .agg(
        related_record_count=("grant_key", "size"),
        related_grant_keys=(
            "grant_key",
            lambda values: " | ".join(
                sorted(values.astype(str))
            ),
        ),
        related_organisations=(
            "organisation_name",
            lambda values: " | ".join(
                sorted(
                    set(
                        values.dropna().astype(str)
                    )
                )
            ),
        ),
    )
)

# Select one deterministic representative record per text group
model_document_df = (
    model_catalog_df.sort_values(
        [
            "text_group_id",
            "award_year",
            "grant_key",
        ],
        ascending=[True, False, True],
    )
    .drop_duplicates(
        subset="text_group_id",
        keep="first",
    )
    .merge(
        text_group_metadata_df,
        on="text_group_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("text_group_id")
    .reset_index(drop=True)
)

document_summary_df = pd.DataFrame(
    {
        "metric": [
            "Full source records",
            "Unique modelling documents",
            "Repeated records collapsed",
            "Duplicate modelling texts",
            "Largest related-record group",
        ],
        "value": [
            len(model_catalog_df),
            len(model_document_df),
            len(model_catalog_df) - len(model_document_df),
            model_document_df["search_text"].duplicated().sum(),
            model_document_df["related_record_count"].max(),
        ],
    }
)

display(document_summary_df)

display(
    model_document_df.loc[
        model_document_df["related_record_count"] > 1,
        [
            "text_group_id",
            "title",
            "related_record_count",
            "related_grant_keys",
            "related_organisations",
        ],
    ].head(10)
)

,metric,value
0,Full source records,3339
1,Unique modelling documents,2943
2,Repeated records collapsed,396
3,Duplicate modelling texts,0
4,Largest related-record group,6


,text_group_id,title,related_record_count,related_grant_keys,related_organisations
38,39,A public workflow for predicting peptide binding structures,2,NSF_2121063 | NSF_2438595,University of Kansas Center for Research Inc | University of North Carolina at Chapel Hill
209,210,BRITE Pivot: Micro-Macro Modeling of Reactive Flow and Rock Weathering Enhanced by Artificial Intelligence,2,NSF_2135584 | NSF_2416344,Cornell University | Georgia Tech Research Corporation
227,228,CAREER: A Multichannel Convolutional Neural Network Framework for Prediction of Damage Nucleation Sites in Microstru...,2,NSF_2142164 | NSF_2341922,Iowa State University | University of Colorado at Colorado Springs
248,249,"CAREER: Assigning comprehensive, standardized sample annotations to enhance the ability to discover, use, and interp...",2,NSF_2045651 | NSF_2328140,Michigan State University | University of Colorado at Denver
252,253,CAREER: Atomistic Investigation of Phase Transition in Nanostructured Silicon--Towards Convergent Understanding with...,2,NSF_2046218 | NSF_2305529,Texas A&M Engineering Experiment Station | University of Texas at San Antonio
267,268,CAREER: Chemical Network Based Understanding and Prediction of Electrolyte Decomposition in Batteries,2,NSF_2045887 | NSF_2526504,Purdue University | University of Notre Dame
294,295,CAREER: Data-Driven Prioritization and Control of Disinfection Byproducts in Drinking Water,2,NSF_2441521 | NSF_2554919,South Dakota School of Mines and Technology | Stevens Institute of Technology
347,348,CAREER: High bandwidth nano-transistors to understand the kinetic basis for CRISPR/CAS enzymes to enhance their appl...,2,NSF_2048283 | NSF_2427540,Keck Graduate Institute | University of California-San Diego
359,360,"CAREER: Integrative Pathway Analysis for Cancer Subtyping, Patient Stratification, and Risk Prediction",2,NSF_2141660 | NSF_2343019,"Auburn University | Board of Regents, NSHE, obo University of Nevada, Reno"
371,372,Career: Learning Multimodal Representations of the Physical World,2,NSF_2339071 | NSF_2611044,Cornell University | Regents of the University of Michigan - Ann Arbor


### Final modelling corpus

The source catalogue retains all 3,339 awards, while the similarity model uses 2,943 unique text documents.

Exact-text duplicates were collapsed to prevent collaborative NSF awards from occupying multiple top-result positions. Their original grant keys and organisations remain available through the group mapping.

The corpus is now ready for the keyword-overlap baseline.

## 2. Keyword-overlap baseline

Before building TF-IDF, a simple keyword baseline will provide a transparent comparison.

The baseline will:

- extract meaningful terms from a research concept;
- remove common English stop words;
- count shared terms with each grant;
- rank the unique modelling documents by overlap.

This approach is easy to explain but does not account for term importance or broader textual context.

In [7]:
# Build a transparent keyword-overlap baseline

import re

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Remove common English words and generic research terminology
baseline_stop_words = set(ENGLISH_STOP_WORDS).union(
    {
        "research",
        "project",
        "study",
        "studies",
        "using",
        "use",
        "used",
        "develop",
        "developing",
        "development",
        "approach",
        "method",
        "methods",
        "new",
    }
)


def tokenize_baseline_text(text):
    """Convert text into normalized word tokens."""

    normalized_text = (
        str(text)
        .lower()
        .replace("-", " ")
    )

    return re.findall(
        r"\b[a-z][a-z0-9]+\b",
        normalized_text,
    )


def extract_query_features(query):
    """Extract meaningful unigrams and two-word phrases."""

    tokens = tokenize_baseline_text(query)

    meaningful_tokens = [
        token
        for token in tokens
        if token not in baseline_stop_words
        and len(token) >= 3
    ]

    query_unigrams = sorted(set(meaningful_tokens))

    query_bigrams = sorted(
        {
            f"{first} {second}"
            for first, second in zip(
                meaningful_tokens,
                meaningful_tokens[1:],
            )
        }
    )

    return query_unigrams, query_bigrams


def keyword_overlap_search(
    query,
    top_n=10,
    source=None,
    topic=None,
):
    """Rank modelling documents using shared words and phrases."""

    if not str(query).strip():
        raise ValueError(
            "The research concept cannot be empty."
        )

    query_unigrams, query_bigrams = (
        extract_query_features(query)
    )

    results_df = model_document_df.copy()

    # Apply optional filters
    if source is not None:
        results_df = results_df.loc[
            results_df["source"] == source
        ].copy()

    if topic is not None:
        results_df = results_df.loc[
            results_df["primary_topic"] == topic
        ].copy()

    if results_df.empty:
        return results_df

    baseline_text = (
        results_df["search_text"]
        .str.replace("-", " ", regex=False)
    )

    results_df["matched_unigrams"] = (
        baseline_text.apply(
            lambda text: [
                term
                for term in query_unigrams
                if re.search(
                    rf"\b{re.escape(term)}\b",
                    text,
                )
            ]
        )
    )

    results_df["matched_phrases"] = (
        baseline_text.apply(
            lambda text: [
                phrase
                for phrase in query_bigrams
                if phrase in text
            ]
        )
    )

    results_df["matched_unigram_count"] = (
        results_df["matched_unigrams"].str.len()
    )

    results_df["matched_phrase_count"] = (
        results_df["matched_phrases"].str.len()
    )

    # Two-word phrases receive slightly more weight
    results_df["baseline_score"] = (
        results_df["matched_unigram_count"]
        + 2 * results_df["matched_phrase_count"]
    )

    results_df = (
        results_df.loc[
            results_df["baseline_score"] > 0
        ]
        .sort_values(
            [
                "baseline_score",
                "matched_phrase_count",
                "matched_unigram_count",
                "award_year",
            ],
            ascending=[False, False, False, False],
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    results_df.insert(
        0,
        "rank",
        range(1, len(results_df) + 1),
    )

    results_df["abstract_preview"] = (
        results_df["abstract"]
        .str.slice(0, 220)
        .str.rstrip()
        + "..."
    )

    return results_df


# Prepared demonstration concept from Notebook 06
demo_query = (
    "Machine-learning-guided discovery of stable heterogeneous "
    "catalysts using automated experimentation and closed-loop "
    "optimisation."
)

demo_keyword_results_df = keyword_overlap_search(
    query=demo_query,
    top_n=10,
)

print("Query features:")
print(extract_query_features(demo_query))

display(
    demo_keyword_results_df[
        [
            "rank",
            "grant_key",
            "source",
            "title",
            "primary_topic",
            "award_year",
            "baseline_score",
            "matched_unigrams",
            "matched_phrases",
            "related_record_count",
            "abstract_preview",
        ]
    ]
)

Query features:
(['automated', 'catalysts', 'closed', 'discovery', 'experimentation', 'guided', 'heterogeneous', 'learning', 'loop', 'machine', 'optimisation', 'stable'], ['automated experimentation', 'catalysts automated', 'closed loop', 'discovery stable', 'experimentation closed', 'guided discovery', 'heterogeneous catalysts', 'learning guided', 'loop optimisation', 'machine learning', 'stable heterogeneous'])


,rank,grant_key,source,title,primary_topic,award_year,baseline_score,matched_unigrams,matched_phrases,related_record_count,abstract_preview
0,1,NSF_2522655,NSF,Collaborative Research: DMREF: NSF-DST: Metastability Engineering in Refractory Multi-Principal Element Alloys,AI-enabled materials,2025,13,"[closed, discovery, experimentation, guided, learning, loop, machine]","[closed loop, guided discovery, machine learning]",2,This Designing Materials to Revolutionize and Engineer our Future (DMREF) joint NSF-Department of Science and Techno...
1,2,NSF_2522658,NSF,Collaborative Research: DMREF: NSF-DST: Metastability Engineering in Refractory Multi-Principal Element Alloys,AI-enabled materials,2025,13,"[closed, discovery, experimentation, guided, learning, loop, machine]","[closed loop, guided discovery, machine learning]",1,This Designing Materials to Revolutionize and Engineer our Future (DMREF) joint NSF-Department of Science and Techno...
2,3,NSF_2522654,NSF,Collaborative Research: DMREF: NSF-DST: Metastability Engineering in Refractory Multi-Principal Element Alloys,AI-enabled materials,2025,13,"[closed, discovery, experimentation, guided, learning, loop, machine]","[closed loop, guided discovery, machine learning]",2,This Designing Materials to Revolutionize and Engineer our Future (DMREF) joint NSF-Department of Science and Techno...
3,4,NSF_2413579,NSF,Collaborative Research: DMREF: Closed-Loop Design of Polymers with Adaptive Networks for Extreme Mechanics,Materials informatics,2024,13,"[automated, closed, discovery, guided, learning, loop, machine]","[closed loop, learning guided, machine learning]",2,"Non-technical Description: Polymer materials such as thermoplastics, thermosets, elastomers, and gels, were produced..."
4,5,NSF_2323729,NSF,Collaborative Research: DMREF: Closed-Loop Design of Polymers with Adaptive Networks for Extreme Mechanics,Materials informatics,2023,13,"[automated, closed, discovery, guided, learning, loop, machine]","[closed loop, learning guided, machine learning]",1,"Non-technical Description: Polymer materials such as thermoplastics, thermosets, elastomers, and gels, were produced..."
5,6,NSF_2306125,NSF,"Collaborative Research: DMREF: Machine Learning-aided Discovery of Synthesizable, Active and Stable Heterogeneous Ca...",AI-enabled catalysis,2022,13,"[catalysts, discovery, heterogeneous, learning, loop, machine, stable]","[heterogeneous catalysts, machine learning, stable heterogeneous]",2,Catalytic materials have long been used to improve the efficiency and product selectivity of many processes of vital...
6,7,NSF_2116646,NSF,"Collaborative Research: DMREF: Machine Learning-aided Discovery of Synthesizable, Active and Stable Heterogeneous Ca...",AI-enabled catalysis,2021,13,"[catalysts, discovery, heterogeneous, learning, loop, machine, stable]","[heterogeneous catalysts, machine learning, stable heterogeneous]",1,Catalytic materials have long been used to improve the efficiency and product selectivity of many processes of vital...
7,8,NSF_2522294,NSF,Collaborative Research: DMREF: Accelerated Discovery and Design of Dynamically Evolving Catalyst Material Surfaces,AI-enabled catalysis,2025,11,"[catalysts, closed, discovery, learning, loop, machine, stable]","[closed loop, machine learning]",1,Catalyst materials that speed up chemical reactions play a critical role in the production of energy and chemicals. ...
8,9,NSF_2522295,NSF,Collaborative Research: DMREF: Accelerated Discovery and Design of Dynamically Evolving Catalyst Material Surfaces,AI-enabled catalysis,2025,11,"[catalysts, closed, discovery, learning, loop, machine, stable]","[closed loop, machine learning]",1,Catalyst materials that speed up chemical reactions play a critical role in the production of energy and chemicals. ...
9,10,NSF_2522293,NSF,Collaborative Research: DMREF: Accelerated Discovery and Design of Dynamically Evolving Catalyst Material Surfaces,AI-enabled catalysis,2025,11,"[catalysts, closed, discovery, learning,

### Baseline result review

The keyword baseline retrieves several relevant catalyst and closed-loop discovery projects.

However, repeated collaborative project families occupy multiple result positions. Exact-text grouping is not sufficient because related awards can contain small metadata or abstract differences.

The search results will therefore be diversified using a normalized project-title key before evaluating the baseline or building the TF-IDF model.

In [8]:
# Create a normalized title key for grouping related project records

def normalize_project_title(title):
    """Normalize titles for result-family grouping."""

    normalized_title = str(title).lower()

    # Remove common administrative prefixes
    normalized_title = re.sub(
        r"^(collaborative research:\s*)+",
        "",
        normalized_title,
    )

    # Remove punctuation and normalize whitespace
    normalized_title = re.sub(
        r"[^a-z0-9\s]",
        " ",
        normalized_title,
    )

    normalized_title = re.sub(
        r"\s+",
        " ",
        normalized_title,
    ).strip()

    return normalized_title


model_document_df["project_title_key"] = (
    model_document_df["title"].apply(
        normalize_project_title
    )
)

print(
    "Unique modelling documents:",
    len(model_document_df),
)

print(
    "Unique normalized project titles:",
    model_document_df["project_title_key"].nunique(),
)

print(
    "Documents sharing a normalized title:",
    model_document_df["project_title_key"]
    .duplicated(keep=False)
    .sum(),
)

display(
    model_document_df.loc[
        model_document_df["project_title_key"]
        .duplicated(keep=False),
        [
            "grant_key",
            "title",
            "source",
            "award_year",
            "project_title_key",
        ],
    ]
    .sort_values(
        ["project_title_key", "award_year"]
    )
    .head(20)
)

Unique modelling documents: 2943
Unique normalized project titles: 2768
Documents sharing a normalized title: 328


,grant_key,title,source,award_year,project_title_key
634,NSF_2435754,Collaborative Research: ACED: Accelerating Protein Engineering with Evolution-Guided Generative AI and a Self-Drivin...,NSF,2025,aced accelerating protein engineering with evolution guided generative ai and a self driving biofoundry
635,NSF_2435755,Collaborative Research: ACED: Accelerating Protein Engineering with Evolution-Guided Generative AI and a Self-Drivin...,NSF,2025,aced accelerating protein engineering with evolution guided generative ai and a self driving biofoundry
637,NSF_2434171,Collaborative Research: ACED: Developing Consistency Model-based Ultra-Long Stride Molecular Simulation to Unravel L...,NSF,2025,aced developing consistency model based ultra long stride molecular simulation to unravel long time dynamics of prot...
638,NSF_2434170,Collaborative Research: ACED: Developing Consistency Model-based Ultra-Long Stride Molecular Simulation to Unravel L...,NSF,2025,aced developing consistency model based ultra long stride molecular simulation to unravel long time dynamics of prot...
647,NSF_2526205,Collaborative Research: Beginnings: Advancing Marine Technology Careers through Immersive Autonomous Underwater Vehi...,NSF,2025,beginnings advancing marine technology careers through immersive autonomous underwater vehicle deployment experiences
648,NSF_2526204,Collaborative Research: Beginnings: Advancing Marine Technology Careers through Immersive Autonomous Underwater Vehi...,NSF,2025,beginnings advancing marine technology careers through immersive autonomous underwater vehicle deployment experiences
166,NSF_2213854,BII: Predicting the global host-virus network from molecular foundations,NSF,2022,bii predicting the global host virus network from molecular foundations
165,NSF_2515340,BII: Predicting the global host-virus network from molecular foundations,NSF,2024,bii predicting the global host virus network from molecular foundations
652,NSF_2319553,Collaborative Research: California-Hawaii Astrophysics Mentoring Partnership (CHAMP),NSF,2023,california hawaii astrophysics mentoring partnership champ
653,NSF_2319554,Collaborative Research: California-Hawaii Astrophysics Mentoring Partnership (CHAMP),NSF,2023,california hawaii astrophysics mentoring partnership champ


In [9]:
# Diversify keyword-baseline results by normalized project title

def keyword_overlap_search(
    query,
    top_n=10,
    source=None,
    topic=None,
    diversify_titles=True,
):
    """Rank modelling documents using shared words and phrases."""

    if not str(query).strip():
        raise ValueError(
            "The research concept cannot be empty."
        )

    query_unigrams, query_bigrams = (
        extract_query_features(query)
    )

    results_df = model_document_df.copy()

    # Apply optional filters
    if source is not None:
        results_df = results_df.loc[
            results_df["source"] == source
        ].copy()

    if topic is not None:
        results_df = results_df.loc[
            results_df["primary_topic"] == topic
        ].copy()

    if results_df.empty:
        return results_df

    baseline_text = (
        results_df["search_text"]
        .str.replace("-", " ", regex=False)
    )

    results_df["matched_unigrams"] = (
        baseline_text.apply(
            lambda text: [
                term
                for term in query_unigrams
                if re.search(
                    rf"\b{re.escape(term)}\b",
                    text,
                )
            ]
        )
    )

    results_df["matched_phrases"] = (
        baseline_text.apply(
            lambda text: [
                phrase
                for phrase in query_bigrams
                if phrase in text
            ]
        )
    )

    results_df["matched_unigram_count"] = (
        results_df["matched_unigrams"].str.len()
    )

    results_df["matched_phrase_count"] = (
        results_df["matched_phrases"].str.len()
    )

    # Two-word phrases receive additional weight
    results_df["baseline_score"] = (
        results_df["matched_unigram_count"]
        + 2 * results_df["matched_phrase_count"]
    )

    # Count how many modelling documents share each title
    title_family_sizes = (
        model_document_df["project_title_key"]
        .value_counts()
    )

    results_df["title_family_document_count"] = (
        results_df["project_title_key"]
        .map(title_family_sizes)
        .fillna(1)
        .astype(int)
    )

    results_df = (
        results_df.loc[
            results_df["baseline_score"] > 0
        ]
        .sort_values(
            [
                "baseline_score",
                "matched_phrase_count",
                "matched_unigram_count",
                "award_year",
                "grant_key",
            ],
            ascending=[False, False, False, False, True],
        )
    )

    # Retain only the highest-ranked document from each project family
    if diversify_titles:
        results_df = results_df.drop_duplicates(
            subset="project_title_key",
            keep="first",
        )

    results_df = (
        results_df.head(top_n)
        .reset_index(drop=True)
    )

    results_df.insert(
        0,
        "rank",
        range(1, len(results_df) + 1),
    )

    results_df["abstract_preview"] = (
        results_df["abstract"]
        .str.slice(0, 220)
        .str.rstrip()
        + "..."
    )

    return results_df


# Rerun the prepared demonstration query
demo_keyword_results_df = keyword_overlap_search(
    query=demo_query,
    top_n=10,
)

display(
    demo_keyword_results_df[
        [
            "rank",
            "grant_key",
            "source",
            "title",
            "primary_topic",
            "award_year",
            "baseline_score",
            "matched_unigrams",
            "matched_phrases",
            "title_family_document_count",
            "related_record_count",
        ]
    ]
)

print(
    "Duplicate project titles in results:",
    demo_keyword_results_df[
        "project_title_key"
    ].duplicated().sum(),
)

,rank,grant_key,source,title,primary_topic,award_year,baseline_score,matched_unigrams,matched_phrases,title_family_document_count,related_record_count
0,1,NSF_2522654,NSF,Collaborative Research: DMREF: NSF-DST: Metastability Engineering in Refractory Multi-Principal Element Alloys,AI-enabled materials,2025,13,"[closed, discovery, experimentation, guided, learning, loop, machine]","[closed loop, guided discovery, machine learning]",3,2
1,2,NSF_2413579,NSF,Collaborative Research: DMREF: Closed-Loop Design of Polymers with Adaptive Networks for Extreme Mechanics,Materials informatics,2024,13,"[automated, closed, discovery, guided, learning, loop, machine]","[closed loop, learning guided, machine learning]",2,2
2,3,NSF_2306125,NSF,"Collaborative Research: DMREF: Machine Learning-aided Discovery of Synthesizable, Active and Stable Heterogeneous Ca...",AI-enabled catalysis,2022,13,"[catalysts, discovery, heterogeneous, learning, loop, machine, stable]","[heterogeneous catalysts, machine learning, stable heterogeneous]",2,2
3,4,NSF_2522293,NSF,Collaborative Research: DMREF: Accelerated Discovery and Design of Dynamically Evolving Catalyst Material Surfaces,AI-enabled catalysis,2025,11,"[catalysts, closed, discovery, learning, loop, machine, stable]","[closed loop, machine learning]",3,1
4,5,NSF_2523281,NSF,Collaborative Research: DMREF: NSF-DFG: NeuroTronics: Designer Doped Semiconductors for Neuromorphic Bioelectronics,AI-enabled materials,2025,11,"[automated, closed, experimentation, learning, loop, machine, stable]","[closed loop, machine learning]",2,1
5,6,NSF_2334969,NSF,Collaborative Research: Beyond the Single-Atom Paradigm: A Priori Design of Dual-Atom Alloy Active Sites for Efficie...,AI-enabled catalysis,2024,11,"[catalysts, discovery, heterogeneous, learning, loop, machine, stable]","[heterogeneous catalysts, machine learning]",1,2
6,7,NSF_2309852,NSF,Semi-Automated Discovery of Synthetic Polymers with Protein Features,AI-enabled catalysis,2023,11,"[automated, catalysts, closed, discovery, learning, loop, machine]","[closed loop, machine learning]",1,1
7,8,NSF_2119103,NSF,DMREF: AI-Guided Accelerated Discovery of Multi-Principal Element Multi-Functional Alloys,Materials informatics,2021,11,"[closed, discovery, guided, learning, loop, machine, stable]","[closed loop, machine learning]",1,1
8,9,NSF_2535176,NSF,CBET-EPSRC: Computationally guided design of novel metal-organic frameworks for enhanced proton conductivity and pho...,AI-enabled catalysis,2025,10,"[discovery, guided, learning, machine]","[guided discovery, learning guided, machine learning]",1,1
9,10,NSF_2522539,NSF,Collaborative Research: DMREF:NSF-NSERC: Data-Driven Multi-Element Doping for Optimally Controlled Ion-Electron Cond...,AI-enabled materials,2025,10,"[automated, closed, discovery, learning, loop, machine]","[closed loop, machine learning]",2,2


Duplicate project titles in results: 0


### Keyword-baseline review

The diversified baseline returns plausible projects without repeating normalized project titles.

- The directly relevant heterogeneous-catalyst discovery project ranks third.
- Closed-loop materials projects rank highly because the baseline rewards shared phrases.
- Some results match the workflow but not the catalyst application.
- All top ten results are NSF records, showing that simple term overlap may favour one source or writing style.

TF-IDF will test whether weighting distinctive terms improves scientific-context ranking.

## 3. TF-IDF similarity model

TF-IDF will represent each grant as a weighted vector of important words and two-word phrases.

Compared with simple keyword overlap, TF-IDF gives less weight to common terms and more weight to distinctive scientific language. Cosine similarity will then measure how closely each grant matches the user’s research concept.

The model will use the 2,943 unique text documents while preserving links to all underlying award records.

In [10]:
# Build the TF-IDF representation of the grant corpus

from sklearn.feature_extraction.text import TfidfVectorizer

# Preserve the exact row order used by the model matrix
model_document_df = model_document_df.reset_index(drop=True)
model_document_df["model_row_id"] = model_document_df.index

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32,
)

grant_tfidf_matrix = tfidf_vectorizer.fit_transform(
    model_document_df["search_text"]
)

# Summarize the fitted model
document_count, vocabulary_size = grant_tfidf_matrix.shape
nonzero_values = grant_tfidf_matrix.nnz

matrix_density_pct = (
    nonzero_values
    / (document_count * vocabulary_size)
    * 100
)

tfidf_summary_df = pd.DataFrame(
    {
        "metric": [
            "Modelling documents",
            "TF-IDF vocabulary size",
            "Matrix rows",
            "Matrix columns",
            "Non-zero values",
            "Matrix density (%)",
            "Duplicate model row IDs",
            "Missing search texts",
        ],
        "value": [
            len(model_document_df),
            len(tfidf_vectorizer.vocabulary_),
            document_count,
            vocabulary_size,
            nonzero_values,
            round(matrix_density_pct, 4),
            model_document_df["model_row_id"].duplicated().sum(),
            model_document_df["search_text"].isna().sum(),
        ],
    }
)

display(tfidf_summary_df)

# Confirm that the query can be transformed using the fitted vocabulary
demo_query_vector = tfidf_vectorizer.transform(
    [demo_query]
)

print("Demonstration query vector shape:", demo_query_vector.shape)
print("Demonstration query non-zero features:", demo_query_vector.nnz)

,metric,value
0,Modelling documents,2943.0000
1,TF-IDF vocabulary size,103411.0000
2,Matrix rows,2943.0000
3,Matrix columns,103411.0000
4,Non-zero values,926662.0000
5,Matrix density (%),0.3045
6,Duplicate model row IDs,0.0000
7,Missing search texts,0.0000


Demonstration query vector shape: (1, 103411)
Demonstration query non-zero features: 23


### TF-IDF representation

The model represents 2,943 unique project documents using 103,411 weighted word and phrase features.

The matrix is highly sparse, which is expected for scientific text. The demonstration query matches 23 model features, confirming that it can be compared with the grant corpus.

The next step is to calculate cosine similarity and return diversified, filterable project recommendations.

In [11]:
# Create a reusable TF-IDF cosine-similarity search function

from sklearn.metrics.pairwise import cosine_similarity


def tfidf_similarity_search(
    query,
    top_n=10,
    source=None,
    topic=None,
    relevance_tier=None,
    year_min=None,
    year_max=None,
    diversify_titles=True,
):
    """Return grants ranked by TF-IDF cosine similarity."""

    if not str(query).strip():
        raise ValueError(
            "The research concept cannot be empty."
        )

    query_vector = tfidf_vectorizer.transform(
        [str(query)]
    )

    if query_vector.nnz == 0:
        raise ValueError(
            "The query does not contain terms recognized "
            "by the fitted TF-IDF model."
        )

    similarity_scores = cosine_similarity(
        query_vector,
        grant_tfidf_matrix,
    ).ravel()

    results_df = model_document_df.copy()

    results_df["similarity_score"] = (
        similarity_scores
    )

    # Apply optional metadata filters
    if source is not None:
        results_df = results_df.loc[
            results_df["source"] == source
        ].copy()

    if topic is not None:
        results_df = results_df.loc[
            results_df["primary_topic"] == topic
        ].copy()

    if relevance_tier is not None:
        results_df = results_df.loc[
            results_df["relevance_tier"]
            == relevance_tier
        ].copy()

    if year_min is not None:
        results_df = results_df.loc[
            results_df["award_year"] >= year_min
        ].copy()

    if year_max is not None:
        results_df = results_df.loc[
            results_df["award_year"] <= year_max
        ].copy()

    results_df = (
        results_df.loc[
            results_df["similarity_score"] > 0
        ]
        .sort_values(
            [
                "similarity_score",
                "award_year",
                "grant_key",
            ],
            ascending=[False, False, True],
        )
    )

    # Show only the highest-ranking record
    # from each normalized project-title family
    if diversify_titles:
        results_df = results_df.drop_duplicates(
            subset="project_title_key",
            keep="first",
        )

    results_df = (
        results_df.head(top_n)
        .reset_index(drop=True)
    )

    if results_df.empty:
        return results_df

    results_df.insert(
        0,
        "rank",
        range(1, len(results_df) + 1),
    )

    results_df["similarity_pct"] = (
        results_df["similarity_score"]
        * 100
    ).round(1)

    results_df["abstract_preview"] = (
        results_df["abstract"]
        .fillna("")
        .str.slice(0, 250)
        .str.rstrip()
        + "..."
    )

    return results_df


# Run the prepared demonstration concept
demo_tfidf_results_df = tfidf_similarity_search(
    query=demo_query,
    top_n=10,
)

display(
    demo_tfidf_results_df[
        [
            "rank",
            "grant_key",
            "source",
            "title",
            "primary_topic",
            "award_year",
            "similarity_pct",
            "programme_name",
            "organisation_name",
            "amount_native",
            "currency",
            "related_record_count",
            "abstract_preview",
            "source_url",
        ]
    ]
)

print(
    "Results returned:",
    len(demo_tfidf_results_df),
)

print(
    "Duplicate project-title families:",
    demo_tfidf_results_df[
        "project_title_key"
    ].duplicated().sum(),
)

,rank,grant_key,source,title,primary_topic,award_year,similarity_pct,programme_name,organisation_name,amount_native,currency,related_record_count,abstract_preview,source_url
0,1,CORDIS_101105235,CORDIS,Computational Studies on Heterogeneous Astrocatalysis of Space-Abundant Transition Metals,AI-enabled catalysis,2024,7.7,MSCA Postdoctoral Fellowships 2022,UNIVERSITAT AUTONOMA DE BARCELONA,165312.96,EUR,1,"The formation of Solar-like planetary systems is a complex process that goes through different steps, where not only...",https://cordis.europa.eu/project/id/101105235
1,2,NSF_2334969,NSF,Collaborative Research: Beyond the Single-Atom Paradigm: A Priori Design of Dual-Atom Alloy Active Sites for Efficie...,AI-enabled catalysis,2024,7.3,CSD-Chem Strcture and Dynamics,Tulane University,340000.00,USD,2,"With support from the Chemical Structure, Dynamics, and Mechanisms A (CSDM-A) program in the Division of Chemistry, ...",https://www.nsf.gov/awardsearch/showAward?AWD_ID=2334969
2,3,CORDIS_101217538,CORDIS,Iktos Robotics: Integrating AI and Robotics for efficient Drug Design and Discovery,AI-enabled chemistry,2025,7.0,Human Centric Generative AI made in Europe,IKTOS,2499616.00,EUR,1,"Drug discovery is a lengthy and costly process, often exceeding a decade and involving investments upwards of €1.3bn...",https://cordis.europa.eu/project/id/101217538
3,4,NSF_2306125,NSF,"Collaborative Research: DMREF: Machine Learning-aided Discovery of Synthesizable, Active and Stable Heterogeneous Ca...",AI-enabled catalysis,2022,6.4,DMREF,Regents of the University of Michigan - Ann Arbor,432455.00,USD,2,Catalytic materials have long been used to improve the efficiency and product selectivity of many processes of vital...,https://www.nsf.gov/awardsearch/showAward?AWD_ID=2306125
4,5,NSF_2231174,NSF,EAGER: ADAPT: Hypotheses Generation in Heterogeneous Catalysis using Causal Inference and Machine Learning,AI-enabled catalysis,2022,6.1,"Chemical Catalysis, Catalysis, OFFICE OF MULTIDISCIPLINARY AC",Regents of the University of Michigan - Ann Arbor,300000.00,USD,1,"With support from the Chemical Catalysis program in the Division of Chemistry (CHE), the Catalysis program from the ...",https://www.nsf.gov/awardsearch/showAward?AWD_ID=2231174
5,6,NSF_2118838,NSF,Collaborative Research: DMREF: Accelerated Data-Driven Discovery of Ion-Conducting Materials,Materials informatics,2021,5.9,DMREF,"University of Maryland, College Park",900000.00,USD,3,NON-TECHNICAL SUMMARY Oxides with fast ion-conduction are crucial components for a wide range of applications includ...,https://www.nsf.gov/awardsearch/showAward?AWD_ID=2118838
6,7,NSF_2320276,NSF,Equipment: MRI: Track 2 Acquisition of an Automated High-Throughput System for Combinatorial Design and Development ...,AI-enabled materials,2023,5.8,"MPS DMR INSTRUMENTATION, Chemical Instrumentation, OFFICE OF MULTIDISCIPLINARY AC, Major Research Instrumentation",University of Illinois at Urbana-Champaign,3596000.00,USD,1,This Major Research Instrumentation (MRI) award supports the acquisition of an automated system for high-throughput ...,https://www.nsf.gov/awardsearch/showAward?AWD_ID=2320276
7,8,NSF_2415023,NSF,Elucidating the role of carrier transport layers on perovskite photovoltaics' stability through automated experiment...,Autonomous laboratories,2024,5.4,"EPMQD: Electronic, Photonic, M",University of California-Davis,425000.00,USD,1,"An emerging class of material, named halide perovskites, have the potential to deliver high-performing and low-cost ...",https://www.nsf.gov/awardsearch/showAward?AWD_ID=2415023
8,9,CORDIS_101206288,CORDIS,Machine Learning-Enhanced Design of Homogeneous Bifunctional Catalysts for CO2 Hydrogenation,AI-enabled catalysis,2025,5.1,MSCA Postdoctoral Fellowships 2024,UNIVERSITETET I OSLO,251578.56,EUR,1,"As a global society, we face the urgent challenge of reducing CO2 emissions. In response, governmental organisations...",https://cordis.europa.eu/project/id/101206288
9,10,CORDIS_19012664

Results returned: 10
Duplicate project-title families: 0


### Initial TF-IDF search

The TF-IDF search returns ten unique project families for the prepared catalysis concept.

The next step adds an explanation for each recommendation by identifying the weighted words and phrases shared between the user query and each retrieved project.

In [12]:
# Explain each TF-IDF recommendation using shared weighted terms

feature_names = tfidf_vectorizer.get_feature_names_out()

# Remove terms that are technically shared but not very informative
explanation_stop_terms = {
    "research",
    "project",
    "study",
    "studies",
    "method",
    "methods",
    "development",
    "develop",
    "using",
    "use",
    "new",
}


def get_shared_tfidf_terms(
    query,
    model_row_id,
    top_k=8,
):
    """Return the strongest TF-IDF features shared by a query and document."""

    query_vector = tfidf_vectorizer.transform(
        [str(query)]
    )

    document_vector = grant_tfidf_matrix[
        int(model_row_id)
    ]

    # Element-wise multiplication keeps only features
    # that occur in both the query and document
    shared_vector = query_vector.multiply(
        document_vector
    ).tocsr()

    if shared_vector.nnz == 0:
        return []

    shared_features = sorted(
        zip(
            shared_vector.indices,
            shared_vector.data,
        ),
        key=lambda item: item[1],
        reverse=True,
    )

    selected_terms = []

    for feature_index, _ in shared_features:
        term = feature_names[feature_index]

        if term in explanation_stop_terms:
            continue

        if term not in selected_terms:
            selected_terms.append(term)

        if len(selected_terms) == top_k:
            break

    return selected_terms


# Add explanation terms to the demonstration results
demo_tfidf_results_df["shared_tfidf_terms"] = (
    demo_tfidf_results_df.apply(
        lambda row: get_shared_tfidf_terms(
            query=demo_query,
            model_row_id=row["model_row_id"],
            top_k=8,
        ),
        axis=1,
    )
)

demo_tfidf_results_df["match_explanation"] = (
    demo_tfidf_results_df[
        "shared_tfidf_terms"
    ].apply(
        lambda terms: " • ".join(terms)
    )
)

display(
    demo_tfidf_results_df[
        [
            "rank",
            "grant_key",
            "source",
            "title",
            "primary_topic",
            "similarity_pct",
            "match_explanation",
        ]
    ]
)

print(
    "Results with explanations:",
    demo_tfidf_results_df[
        "shared_tfidf_terms"
    ].str.len().gt(0).sum(),
    "/",
    len(demo_tfidf_results_df),
)

,rank,grant_key,source,title,primary_topic,similarity_pct,match_explanation
0,1,CORDIS_101105235,CORDIS,Computational Studies on Heterogeneous Astrocatalysis of Space-Abundant Transition Metals,AI-enabled catalysis,7.7,catalysts using • heterogeneous catalysts • heterogeneous • catalysts • machine learning • machine • learning
1,2,NSF_2334969,NSF,Collaborative Research: Beyond the Single-Atom Paradigm: A Priori Design of Dual-Atom Alloy Active Sites for Efficie...,AI-enabled catalysis,7.3,catalysts using • heterogeneous catalysts • catalysts • stable • heterogeneous • loop • discovery • machine learning
2,3,CORDIS_101217538,CORDIS,Iktos Robotics: Integrating AI and Robotics for efficient Drug Design and Discovery,AI-enabled chemistry,7.0,closed loop • closed • loop • discovery • guided • automated
3,4,NSF_2306125,NSF,"Collaborative Research: DMREF: Machine Learning-aided Discovery of Synthesizable, Active and Stable Heterogeneous Ca...",AI-enabled catalysis,6.4,stable heterogeneous • heterogeneous catalysts • catalysts • stable • loop • heterogeneous • discovery • machine lea...
4,5,NSF_2231174,NSF,EAGER: ADAPT: Hypotheses Generation in Heterogeneous Catalysis using Causal Inference and Machine Learning,AI-enabled catalysis,6.1,heterogeneous catalysts • heterogeneous • catalysts • discovery • machine learning • machine • learning
5,6,NSF_2118838,NSF,Collaborative Research: DMREF: Accelerated Data-Driven Discovery of Ion-Conducting Materials,Materials informatics,5.9,closed loop • closed • loop • discovery
6,7,NSF_2320276,NSF,Equipment: MRI: Track 2 Acquisition of an Automated High-Throughput System for Combinatorial Design and Development ...,AI-enabled materials,5.8,closed loop • automated • closed • loop • discovery • guided
7,8,NSF_2415023,NSF,Elucidating the role of carrier transport layers on perovskite photovoltaics' stability through automated experiment...,Autonomous laboratories,5.4,automated experimentation • stable • automated • experimentation • machine learning • machine • learning
8,9,CORDIS_101206288,CORDIS,Machine Learning-Enhanced Design of Homogeneous Bifunctional Catalysts for CO2 Hydrogenation,AI-enabled catalysis,5.1,catalysts • heterogeneous catalysts • heterogeneous • machine learning • machine • learning
9,10,CORDIS_190126641,CORDIS,Closed-loop deep learning in early-stage drug discovery - cloud platform for targeted protein degradation,AI-enabled chemistry,4.9,closed loop • closed • loop • automated • discovery • learning


Results with explanations: 10 / 10


### Recommendation explainability

All ten TF-IDF results include shared weighted words or phrases explaining the match.

These terms make the ranking easier to inspect and can later appear as the `Why it matched` field in Streamlit.

The next step is to compare the ranking with the manually reviewed evaluation set.

In [13]:
# Compare the TF-IDF ranking with the manually reviewed reference set

# Calculate similarity scores for the full modelling corpus
evaluation_query_vector = tfidf_vectorizer.transform(
    [demo_query]
)

evaluation_similarity_scores = cosine_similarity(
    evaluation_query_vector,
    grant_tfidf_matrix,
).ravel()

full_tfidf_ranking_df = model_document_df.copy()

full_tfidf_ranking_df["similarity_score"] = (
    evaluation_similarity_scores
)

# Rank one representative from each normalized project-title family
full_tfidf_ranking_df = (
    full_tfidf_ranking_df.sort_values(
        [
            "similarity_score",
            "award_year",
            "grant_key",
        ],
        ascending=[False, False, True],
    )
    .drop_duplicates(
        subset="project_title_key",
        keep="first",
    )
    .reset_index(drop=True)
)

full_tfidf_ranking_df["tfidf_rank"] = (
    full_tfidf_ranking_df.index + 1
)

full_tfidf_ranking_df["similarity_pct"] = (
    full_tfidf_ranking_df["similarity_score"]
    * 100
).round(1)

# Create the same normalized title-family key for benchmark records
tfidf_evaluation_df = evaluation_reference_df.copy()

tfidf_evaluation_df["project_title_key"] = (
    tfidf_evaluation_df["title"].apply(
        normalize_project_title
    )
)

# Attach each benchmark project's TF-IDF rank
ranking_lookup_df = full_tfidf_ranking_df[
    [
        "project_title_key",
        "grant_key",
        "tfidf_rank",
        "similarity_pct",
    ]
].rename(
    columns={
        "grant_key": "ranked_representative_grant_key",
    }
)

tfidf_evaluation_df = tfidf_evaluation_df.merge(
    ranking_lookup_df,
    on="project_title_key",
    how="left",
    validate="many_to_one",
)

# Add simple retrieval indicators
tfidf_evaluation_df["retrieved_top_10"] = (
    tfidf_evaluation_df["tfidf_rank"] <= 10
)

tfidf_evaluation_df["retrieved_top_25"] = (
    tfidf_evaluation_df["tfidf_rank"] <= 25
)

tfidf_evaluation_df["retrieved_top_50"] = (
    tfidf_evaluation_df["tfidf_rank"] <= 50
)

tfidf_evaluation_df = (
    tfidf_evaluation_df.sort_values(
        [
            "manual_relevance_score",
            "tfidf_rank",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(
    tfidf_evaluation_df[
        [
            "grant_key",
            "title",
            "manual_relevance_score",
            "manual_relevance_label",
            "tfidf_rank",
            "similarity_pct",
            "retrieved_top_10",
            "retrieved_top_25",
            "retrieved_top_50",
            "review_notes",
        ]
    ]
)

# Summarize known-reference retrieval
relevant_reference_mask = (
    tfidf_evaluation_df["manual_relevance_score"] >= 2
)

evaluation_summary_df = pd.DataFrame(
    {
        "metric": [
            "Benchmark projects",
            "Projects mapped to ranking",
            "Relevant or strong projects",
            "Relevant or strong in top 10",
            "Relevant or strong in top 25",
            "Relevant or strong in top 50",
            "Strong matches in top 10",
            "Weak-match TF-IDF rank",
        ],
        "value": [
            len(tfidf_evaluation_df),
            tfidf_evaluation_df["tfidf_rank"].notna().sum(),
            relevant_reference_mask.sum(),
            (
                relevant_reference_mask
                & tfidf_evaluation_df["retrieved_top_10"]
            ).sum(),
            (
                relevant_reference_mask
                & tfidf_evaluation_df["retrieved_top_25"]
            ).sum(),
            (
                relevant_reference_mask
                & tfidf_evaluation_df["retrieved_top_50"]
            ).sum(),
            (
                (tfidf_evaluation_df["manual_relevance_score"] == 3)
                & tfidf_evaluation_df["retrieved_top_10"]
            ).sum(),
            tfidf_evaluation_df.loc[
                tfidf_evaluation_df["manual_relevance_score"] == 1,
                "tfidf_rank",
            ].min(),
        ],
    }
)

display(evaluation_summary_df)

,grant_key,title,manual_relevance_score,manual_relevance_label,tfidf_rank,similarity_pct,retrieved_top_10,retrieved_top_25,retrieved_top_50,review_notes
0,NSF_2554343,Collaborative Research: DMREF: Atomically precise catalyst design for selective bond activation,3,Strong match,54,3.3,False,False,False,Machine-learning-guided heterogeneous catalyst design closely matches the scientific component of the proposed concept.
1,CORDIS_101206634,Robot-mediated development of statistical models for mechanistic analysis amplification in synthetic organic reactions,3,Strong match,269,1.6,False,False,False,"Robot-mediated experimentation, machine learning, reaction optimisation, and mechanistic analysis closely match the ..."
2,NSF_2309852,Semi-Automated Discovery of Synthetic Polymers with Protein Features,2,Relevant,12,4.8,False,True,True,"Strong semi-automated and closed-loop discovery workflow, although the scientific focus is synthetic polymers rather..."
3,CORDIS_101118768,Directed Evolution of Metastable Electrocatalyst Interfaces for Energy Conversion,2,Relevant,52,3.4,False,False,False,"Strong catalyst-design relevance and machine-learning content, but automation and closed-loop experimentation are le..."
4,NSF_2318141,CCI Phase I: NSF Center for Sustainable Photoredox Catalysis (SuPRCat),2,Relevant,76,2.9,False,False,False,"Strong data-driven and sustainable catalysis relevance, but closed-loop automated experimentation is less explicit."
5,CORDIS_101062692,Computationally driven discovery of organic dyes for photoredox catalysis from physicochemical principles and mechan...,2,Relevant,442,1.2,False,False,False,"Computational and machine-learning-guided photoredox catalyst discovery, but limited evidence of closed-loop experim..."
6,CORDIS_101204747,Development of Data-assisted Photo-Organocatalytic Transformations,2,Relevant,1113,0.6,False,False,False,"Data-assisted photocatalytic reaction development is relevant, but the automation component appears limited."
7,CORDIS_101098001,"Automated, miniaturized and accelerated drug discovery: AMADEUS",1,Weak match,138,2.2,False,False,False,"Strong automation and machine-learning overlap, but the main application is drug discovery rather than catalyst disc..."


,metric,value
0,Benchmark projects,8
1,Projects mapped to ranking,8
2,Relevant or strong projects,7
3,Relevant or strong in top 10,0
4,Relevant or strong in top 25,1
5,Relevant or strong in top 50,1
6,Strong matches in top 10,0
7,Weak-match TF-IDF rank,138


### Initial model evaluation

The first TF-IDF model retrieves scientifically related projects, but its ranking does not align closely enough with the manually reviewed benchmark.

- All eight benchmark projects were successfully mapped.
- Only one of seven relevant or strong projects appeared in the top 25.
- A strong catalyst-design match ranked 54th.
- Several relevant CORDIS projects ranked substantially lower.
- The weak drug-discovery comparison ranked above some stronger catalyst projects.

This suggests that long abstracts and source-specific writing styles dilute the most important scientific concepts. The model will therefore be improved by normalizing terminology and giving project titles greater weight before repeating the evaluation.

In [14]:
# Normalize terminology and create a title-weighted model text

def normalize_model_text(text):
    """Standardize selected terminology used across grant sources."""

    text = str(text).lower()

    # Standardize punctuation and common multi-word expressions
    text = re.sub(r"machine[-\s]learning", "machine learning", text)
    text = re.sub(r"closed[-\s]loop", "closed loop", text)
    text = re.sub(r"self[-\s]driving", "self driving autonomous", text)
    text = re.sub(r"data[-\s]driven", "data driven", text)
    text = re.sub(r"ai[-\s]guided", "ai guided", text)

    # Normalize British and American spelling
    text = re.sub(r"\boptimisation\b", "optimization", text)
    text = re.sub(r"\boptimise\b", "optimize", text)
    text = re.sub(r"\boptimised\b", "optimized", text)
    text = re.sub(r"\bmodelling\b", "modeling", text)

    # Normalize related scientific word forms
    text = re.sub(
        r"\b(catalysts|catalytic|catalysis)\b",
        "catalyst",
        text,
    )

    text = re.sub(
        r"\b(automated|automating|automation)\b",
        "automation",
        text,
    )

    text = re.sub(
        r"\b(robotic|robotics|robot-mediated)\b",
        "robot",
        text,
    )

    text = re.sub(
        r"\b(experiments|experimental)\b",
        "experimentation",
        text,
    )

    # Normalize remaining punctuation and whitespace
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


model_document_df["normalized_title"] = (
    model_document_df["title_clean"]
    .apply(normalize_model_text)
)

model_document_df["normalized_abstract"] = (
    model_document_df["abstract_clean"]
    .apply(normalize_model_text)
)

model_document_df["normalized_programme"] = (
    model_document_df["programme_name_clean"]
    .apply(normalize_model_text)
)

# Repeat titles three times so central project concepts
# have more influence than incidental abstract language
model_document_df["weighted_search_text"] = (
    model_document_df["normalized_title"]
    + " "
    + model_document_df["normalized_title"]
    + " "
    + model_document_df["normalized_title"]
    + " "
    + model_document_df["normalized_abstract"]
    + " "
    + model_document_df["normalized_programme"]
).str.strip()

normalization_summary_df = pd.DataFrame(
    {
        "metric": [
            "Model documents",
            "Missing weighted texts",
            "Empty weighted texts",
            "Median weighted-text words",
            "Minimum weighted-text words",
            "Maximum weighted-text words",
        ],
        "value": [
            len(model_document_df),
            model_document_df[
                "weighted_search_text"
            ].isna().sum(),
            model_document_df[
                "weighted_search_text"
            ].eq("").sum(),
            model_document_df[
                "weighted_search_text"
            ].str.split().str.len().median(),
            model_document_df[
                "weighted_search_text"
            ].str.split().str.len().min(),
            model_document_df[
                "weighted_search_text"
            ].str.split().str.len().max(),
        ],
    }
)

display(normalization_summary_df)

print("\nOriginal demonstration query:")
print(demo_query)

print("\nNormalized demonstration query:")
print(normalize_model_text(demo_query))

,metric,value
0,Model documents,2943.0
1,Missing weighted texts,0.0
2,Empty weighted texts,0.0
3,Median weighted-text words,442.0
4,Minimum weighted-text words,119.0
5,Maximum weighted-text words,1200.0



Original demonstration query:
Machine-learning-guided discovery of stable heterogeneous catalysts using automated experimentation and closed-loop optimisation.

Normalized demonstration query:
machine learning guided discovery of stable heterogeneous catalyst using automation experimentation and closed loop optimization


In [15]:
# Build the improved TF-IDF model using normalized, title-weighted text

improved_tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32,
)

improved_grant_tfidf_matrix = (
    improved_tfidf_vectorizer.fit_transform(
        model_document_df["weighted_search_text"]
    )
)

normalized_demo_query = normalize_model_text(
    demo_query
)

improved_demo_query_vector = (
    improved_tfidf_vectorizer.transform(
        [normalized_demo_query]
    )
)

document_count, vocabulary_size = (
    improved_grant_tfidf_matrix.shape
)

matrix_density_pct = (
    improved_grant_tfidf_matrix.nnz
    / (document_count * vocabulary_size)
    * 100
)

improved_model_summary_df = pd.DataFrame(
    {
        "metric": [
            "Modelling documents",
            "Vocabulary size",
            "Matrix rows",
            "Matrix columns",
            "Non-zero values",
            "Matrix density (%)",
            "Query non-zero features",
            "Duplicate model row IDs",
        ],
        "value": [
            len(model_document_df),
            len(
                improved_tfidf_vectorizer.vocabulary_
            ),
            document_count,
            vocabulary_size,
            improved_grant_tfidf_matrix.nnz,
            round(matrix_density_pct, 4),
            improved_demo_query_vector.nnz,
            model_document_df[
                "model_row_id"
            ].duplicated().sum(),
        ],
    }
)

display(improved_model_summary_df)

,metric,value
0,Modelling documents,2943.000
1,Vocabulary size,103714.000
2,Matrix rows,2943.000
3,Matrix columns,103714.000
4,Non-zero values,927882.000
5,Matrix density (%),0.304
6,Query non-zero features,24.000
7,Duplicate model row IDs,0.000


In [16]:
# Search the improved TF-IDF model

def improved_tfidf_similarity_search(
    query,
    top_n=10,
    source=None,
    topic=None,
    relevance_tier=None,
    year_min=None,
    year_max=None,
    diversify_titles=True,
):
    """Return projects ranked by improved TF-IDF similarity."""

    if not str(query).strip():
        raise ValueError(
            "The research concept cannot be empty."
        )

    normalized_query = normalize_model_text(query)

    query_vector = (
        improved_tfidf_vectorizer.transform(
            [normalized_query]
        )
    )

    if query_vector.nnz == 0:
        raise ValueError(
            "The query contains no terms recognized "
            "by the improved TF-IDF model."
        )

    similarity_scores = cosine_similarity(
        query_vector,
        improved_grant_tfidf_matrix,
    ).ravel()

    results_df = model_document_df.copy()

    results_df["similarity_score"] = (
        similarity_scores
    )

    # Optional metadata filters
    if source is not None:
        results_df = results_df.loc[
            results_df["source"] == source
        ].copy()

    if topic is not None:
        results_df = results_df.loc[
            results_df["primary_topic"] == topic
        ].copy()

    if relevance_tier is not None:
        results_df = results_df.loc[
            results_df["relevance_tier"]
            == relevance_tier
        ].copy()

    if year_min is not None:
        results_df = results_df.loc[
            results_df["award_year"] >= year_min
        ].copy()

    if year_max is not None:
        results_df = results_df.loc[
            results_df["award_year"] <= year_max
        ].copy()

    results_df = (
        results_df.loc[
            results_df["similarity_score"] > 0
        ]
        .sort_values(
            [
                "similarity_score",
                "award_year",
                "grant_key",
            ],
            ascending=[False, False, True],
        )
    )

    if diversify_titles:
        results_df = results_df.drop_duplicates(
            subset="project_title_key",
            keep="first",
        )

    results_df = (
        results_df.head(top_n)
        .reset_index(drop=True)
    )

    results_df.insert(
        0,
        "rank",
        range(1, len(results_df) + 1),
    )

    results_df["similarity_pct"] = (
        results_df["similarity_score"]
        * 100
    ).round(1)

    results_df["abstract_preview"] = (
        results_df["abstract"]
        .fillna("")
        .str.slice(0, 220)
        .str.rstrip()
        + "..."
    )

    return results_df


improved_demo_results_df = (
    improved_tfidf_similarity_search(
        query=demo_query,
        top_n=10,
    )
)

display(
    improved_demo_results_df[
        [
            "rank",
            "grant_key",
            "source",
            "title",
            "primary_topic",
            "award_year",
            "similarity_pct",
            "programme_name",
            "organisation_name",
            "related_record_count",
            "abstract_preview",
        ]
    ]
)

print(
    "Results returned:",
    len(improved_demo_results_df),
)

print(
    "Duplicate project-title families:",
    improved_demo_results_df[
        "project_title_key"
    ].duplicated().sum(),
)

,rank,grant_key,source,title,primary_topic,award_year,similarity_pct,programme_name,organisation_name,related_record_count,abstract_preview
0,1,NSF_2231174,NSF,EAGER: ADAPT: Hypotheses Generation in Heterogeneous Catalysis using Causal Inference and Machine Learning,AI-enabled catalysis,2022,11.2,"Chemical Catalysis, Catalysis, OFFICE OF MULTIDISCIPLINARY AC",Regents of the University of Michigan - Ann Arbor,1,"With support from the Chemical Catalysis program in the Division of Chemistry (CHE), the Catalysis program from the ..."
1,2,NSF_2306125,NSF,"Collaborative Research: DMREF: Machine Learning-aided Discovery of Synthesizable, Active and Stable Heterogeneous Ca...",AI-enabled catalysis,2022,10.2,DMREF,Regents of the University of Michigan - Ann Arbor,2,Catalytic materials have long been used to improve the efficiency and product selectivity of many processes of vital...
2,3,NSF_2339026,NSF,CAREER: Learning mechanistic models with automated experiments,Autonomous laboratories,2024,9.3,"Cross-BIO Activities, Systems and Synthetic Biology",Regents of the University of Michigan - Ann Arbor,1,"While the microbiome revolution revealed thousands of new species, it also presents a challenge as new species are i..."
3,4,NSF_2413579,NSF,Collaborative Research: DMREF: Closed-Loop Design of Polymers with Adaptive Networks for Extreme Mechanics,Materials informatics,2024,8.6,DMREF,Washington University,2,"Non-technical Description: Polymer materials such as thermoplastics, thermosets, elastomers, and gels, were produced..."
4,5,CORDIS_101105235,CORDIS,Computational Studies on Heterogeneous Astrocatalysis of Space-Abundant Transition Metals,AI-enabled catalysis,2024,8.5,MSCA Postdoctoral Fellowships 2022,UNIVERSITAT AUTONOMA DE BARCELONA,1,"The formation of Solar-like planetary systems is a complex process that goes through different steps, where not only..."
5,6,CORDIS_190126641,CORDIS,Closed-loop deep learning in early-stage drug discovery - cloud platform for targeted protein degradation,AI-enabled chemistry,2022,7.8,EIC Accelerator Challenge: Technologies for Open Strategic Autonomy,CELERIS THERAPEUTICS GMBH,1,"Celeris Therapeutics is a deep learning company that uses innovative, in-silico methods such as geometric deep learn..."
6,7,NSF_2409631,NSF,Conference: Artificial Intelligence for Multidisciplinary Exploration and Discovery (AIMED) in Heterogeneous Catalys...,AI-enabled catalysis,2024,7.7,"Chemical Catalysis, Catalysis",Virginia Polytechnic Institute and State University,1,The Artificial Intelligence for Multidisciplinary Exploration and Discovery in Heterogeneous Catalysis Workshop (AIM...
7,8,NSF_2324157,NSF,Collaborative Research: DMREF: Computationally Driven Discovery and Synthesis of 2D Materials through Selective Etching,AI-enabled catalysis,2023,7.5,CERAMICS,Tuskegee University,2,Non-technical Description: The discovery of novel two-dimensional (2D) materials is a very attractive research direc...
8,9,NSF_2203354,NSF,Emergence of Structure and Function from Sequenceable Sequence-Defined Macrocyclic Oligourethanes,AI-enabled catalysis,2022,6.9,"Macromolec/Supramolec/Nano, Chemical Catalysis, Chemical Synthesis",University of Texas at Austin,1,With the support from the Chemical Catalysis (CAT) program and co-funding from the Chemical Synthesis (SYN) and Macr...
9,10,CORDIS_101217538,CORDIS,Iktos Robotics: Integrating AI and Robotics for efficient Drug Design and Discovery,AI-enabled chemistry,2025,6.7,Human Centric Generative AI made in Europe,IKTOS,1,"Drug discovery is a lengthy and costly process, often exceeding a decade and involving investments upwards of €1.3bn..."


Results returned: 10
Duplicate project-title families: 0


In [17]:
# Evaluate the improved TF-IDF model against the manual reference set

improved_evaluation_query_vector = (
    improved_tfidf_vectorizer.transform(
        [normalize_model_text(demo_query)]
    )
)

improved_similarity_scores = cosine_similarity(
    improved_evaluation_query_vector,
    improved_grant_tfidf_matrix,
).ravel()

improved_full_ranking_df = model_document_df.copy()

improved_full_ranking_df["improved_similarity_score"] = (
    improved_similarity_scores
)

# Rank one representative from each project-title family
improved_full_ranking_df = (
    improved_full_ranking_df
    .sort_values(
        [
            "improved_similarity_score",
            "award_year",
            "grant_key",
        ],
        ascending=[False, False, True],
    )
    .drop_duplicates(
        subset="project_title_key",
        keep="first",
    )
    .reset_index(drop=True)
)

improved_full_ranking_df["improved_tfidf_rank"] = (
    improved_full_ranking_df.index + 1
)

improved_full_ranking_df["improved_similarity_pct"] = (
    improved_full_ranking_df[
        "improved_similarity_score"
    ]
    * 100
).round(1)

improved_ranking_lookup_df = (
    improved_full_ranking_df[
        [
            "project_title_key",
            "grant_key",
            "improved_tfidf_rank",
            "improved_similarity_pct",
        ]
    ]
    .rename(
        columns={
            "grant_key":
                "improved_representative_grant_key",
        }
    )
)

# Begin with the earlier evaluation results
improved_evaluation_df = (
    tfidf_evaluation_df.copy()
)

improved_evaluation_df = (
    improved_evaluation_df.merge(
        improved_ranking_lookup_df,
        on="project_title_key",
        how="left",
        validate="many_to_one",
    )
)

# Positive values mean the project moved upward
improved_evaluation_df["rank_improvement"] = (
    improved_evaluation_df["tfidf_rank"]
    - improved_evaluation_df["improved_tfidf_rank"]
)

improved_evaluation_df["improved_top_10"] = (
    improved_evaluation_df[
        "improved_tfidf_rank"
    ] <= 10
)

improved_evaluation_df["improved_top_25"] = (
    improved_evaluation_df[
        "improved_tfidf_rank"
    ] <= 25
)

improved_evaluation_df["improved_top_50"] = (
    improved_evaluation_df[
        "improved_tfidf_rank"
    ] <= 50
)

improved_evaluation_df = (
    improved_evaluation_df.sort_values(
        [
            "manual_relevance_score",
            "improved_tfidf_rank",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(
    improved_evaluation_df[
        [
            "grant_key",
            "title",
            "manual_relevance_label",
            "tfidf_rank",
            "improved_tfidf_rank",
            "rank_improvement",
            "similarity_pct",
            "improved_similarity_pct",
            "improved_top_10",
            "improved_top_25",
            "improved_top_50",
        ]
    ]
)

relevant_mask = (
    improved_evaluation_df[
        "manual_relevance_score"
    ] >= 2
)

improved_evaluation_summary_df = pd.DataFrame(
    {
        "metric": [
            "Relevant or strong projects",
            "Relevant or strong in top 10",
            "Relevant or strong in top 25",
            "Relevant or strong in top 50",
            "Strong matches in top 10",
            "Median rank improvement",
            "Weak-match improved rank",
        ],
        "value": [
            relevant_mask.sum(),
            (
                relevant_mask
                & improved_evaluation_df[
                    "improved_top_10"
                ]
            ).sum(),
            (
                relevant_mask
                & improved_evaluation_df[
                    "improved_top_25"
                ]
            ).sum(),
            (
                relevant_mask
                & improved_evaluation_df[
                    "improved_top_50"
                ]
            ).sum(),
            (
                (
                    improved_evaluation_df[
                        "manual_relevance_score"
                    ] == 3
                )
                & improved_evaluation_df[
                    "improved_top_10"
                ]
            ).sum(),
            improved_evaluation_df[
                "rank_improvement"
            ].median(),
            improved_evaluation_df.loc[
                improved_evaluation_df[
                    "manual_relevance_score"
                ] == 1,
                "improved_tfidf_rank",
            ].min(),
        ],
    }
)

display(improved_evaluation_summary_df)

,grant_key,title,manual_relevance_label,tfidf_rank,improved_tfidf_rank,rank_improvement,similarity_pct,improved_similarity_pct,improved_top_10,improved_top_25,improved_top_50
0,NSF_2554343,Collaborative Research: DMREF: Atomically precise catalyst design for selective bond activation,Strong match,54,41,13,3.3,4.4,False,False,True
1,CORDIS_101206634,Robot-mediated development of statistical models for mechanistic analysis amplification in synthetic organic reactions,Strong match,269,178,91,1.6,2.3,False,False,False
2,NSF_2309852,Semi-Automated Discovery of Synthetic Polymers with Protein Features,Relevant,12,24,-12,4.8,5.6,False,True,True
3,NSF_2318141,CCI Phase I: NSF Center for Sustainable Photoredox Catalysis (SuPRCat),Relevant,76,79,-3,2.9,3.2,False,False,False
4,CORDIS_101118768,Directed Evolution of Metastable Electrocatalyst Interfaces for Energy Conversion,Relevant,52,93,-41,3.4,3.0,False,False,False
5,CORDIS_101062692,Computationally driven discovery of organic dyes for photoredox catalysis from physicochemical principles and mechan...,Relevant,442,99,343,1.2,3.0,False,False,False
6,CORDIS_101204747,Development of Data-assisted Photo-Organocatalytic Transformations,Relevant,1113,371,742,0.6,1.6,False,False,False
7,CORDIS_101098001,"Automated, miniaturized and accelerated drug discovery: AMADEUS",Weak match,138,54,84,2.2,3.9,False,False,False


,metric,value
0,Relevant or strong projects,7.0
1,Relevant or strong in top 10,0.0
2,Relevant or strong in top 25,1.0
3,Relevant or strong in top 50,2.0
4,Strong matches in top 10,0.0
5,Median rank improvement,48.5
6,Weak-match improved rank,54.0


In [18]:
# Build a hybrid ranking using TF-IDF plus concept coverage

DOMAIN_CONCEPT_LEXICON = {
    "catalysis": (
        "catalyst",
        "electrocatalyst",
        "photocatalyst",
        "organocatalyst",
    ),
    "materials": (
        "material",
        "alloy",
        "semiconductor",
        "framework",
        "composite",
        "polymer",
    ),
    "chemistry": (
        "chemical",
        "chemistry",
        "synthesis",
        "synthetic",
        "molecule",
        "molecular",
    ),
    "reactions": (
        "reaction",
        "reactivity",
        "mechanism",
        "bond activation",
    ),
    "biology": (
        "protein",
        "enzyme",
        "biological",
        "biomolecule",
    ),
    "drug discovery": (
        "drug",
        "therapeutic",
        "pharmaceutical",
    ),
}

WORKFLOW_CONCEPT_LEXICON = {
    "machine learning": (
        "machine learning",
        "artificial intelligence",
        "deep learning",
        "neural network",
        "data driven",
    ),
    "automation": (
        "automation",
        "autonomous",
        "robot",
        "self driving",
    ),
    "experimentation": (
        "experimentation",
        "experiment",
    ),
    "closed loop": (
        "closed loop",
    ),
    "optimization": (
        "optimization",
        "optimize",
        "screening",
    ),
    "discovery": (
        "discovery",
        "design",
    ),
}


def contains_term(text, term):
    """Check whether a normalized word or phrase occurs in text."""

    pattern = rf"\b{re.escape(term)}\b"

    return bool(
        re.search(
            pattern,
            str(text),
        )
    )


def detect_query_concepts(
    query,
    concept_lexicon,
):
    """Identify controlled concepts represented in a query."""

    normalized_query = normalize_model_text(query)

    return [
        concept_name
        for concept_name, terms in concept_lexicon.items()
        if any(
            contains_term(
                normalized_query,
                term,
            )
            for term in terms
        )
    ]


def calculate_concept_coverage(
    document_text,
    active_concepts,
    concept_lexicon,
):
    """Calculate the share of query concepts found in a document."""

    if not active_concepts:
        return 0.0

    matched_concepts = 0

    for concept_name in active_concepts:
        concept_terms = concept_lexicon[
            concept_name
        ]

        if any(
            contains_term(
                document_text,
                term,
            )
            for term in concept_terms
        ):
            matched_concepts += 1

    return matched_concepts / len(active_concepts)


def hybrid_similarity_search(
    query,
    top_n=10,
    source=None,
    topic=None,
    relevance_tier=None,
    year_min=None,
    year_max=None,
    diversify_titles=True,
):
    """Rank projects using TF-IDF and query-concept coverage."""

    if not str(query).strip():
        raise ValueError(
            "The research concept cannot be empty."
        )

    normalized_query = normalize_model_text(
        query
    )

    query_vector = (
        improved_tfidf_vectorizer.transform(
            [normalized_query]
        )
    )

    similarity_scores = cosine_similarity(
        query_vector,
        improved_grant_tfidf_matrix,
    ).ravel()

    results_df = model_document_df.copy()

    results_df["similarity_score"] = (
        similarity_scores
    )

    # Optional metadata filters
    if source is not None:
        results_df = results_df.loc[
            results_df["source"] == source
        ].copy()

    if topic is not None:
        results_df = results_df.loc[
            results_df["primary_topic"] == topic
        ].copy()

    if relevance_tier is not None:
        results_df = results_df.loc[
            results_df["relevance_tier"]
            == relevance_tier
        ].copy()

    if year_min is not None:
        results_df = results_df.loc[
            results_df["award_year"] >= year_min
        ].copy()

    if year_max is not None:
        results_df = results_df.loc[
            results_df["award_year"] <= year_max
        ].copy()

    results_df = results_df.loc[
        results_df["similarity_score"] > 0
    ].copy()

    if results_df.empty:
        return results_df

    active_domain_concepts = (
        detect_query_concepts(
            query,
            DOMAIN_CONCEPT_LEXICON,
        )
    )

    active_workflow_concepts = (
        detect_query_concepts(
            query,
            WORKFLOW_CONCEPT_LEXICON,
        )
    )

    # Normalize TF-IDF relative to the strongest result
    results_df["tfidf_relative_score"] = (
        results_df["similarity_score"]
        / results_df["similarity_score"].max()
    )

    results_df["domain_coverage"] = (
        results_df["weighted_search_text"].apply(
            lambda text: calculate_concept_coverage(
                text,
                active_domain_concepts,
                DOMAIN_CONCEPT_LEXICON,
            )
        )
    )

    results_df["workflow_coverage"] = (
        results_df["weighted_search_text"].apply(
            lambda text: calculate_concept_coverage(
                text,
                active_workflow_concepts,
                WORKFLOW_CONCEPT_LEXICON,
            )
        )
    )

    # Dynamically redistribute weights when a query
    # lacks domain or workflow concepts
    if (
        active_domain_concepts
        and active_workflow_concepts
    ):
        tfidf_weight = 0.70
        domain_weight = 0.20
        workflow_weight = 0.10

    elif active_domain_concepts:
        tfidf_weight = 0.80
        domain_weight = 0.20
        workflow_weight = 0.00

    elif active_workflow_concepts:
        tfidf_weight = 0.85
        domain_weight = 0.00
        workflow_weight = 0.15

    else:
        tfidf_weight = 1.00
        domain_weight = 0.00
        workflow_weight = 0.00

    results_df["hybrid_score"] = (
        tfidf_weight
        * results_df["tfidf_relative_score"]
        + domain_weight
        * results_df["domain_coverage"]
        + workflow_weight
        * results_df["workflow_coverage"]
    )

    results_df = results_df.sort_values(
        [
            "hybrid_score",
            "similarity_score",
            "award_year",
            "grant_key",
        ],
        ascending=[False, False, False, True],
    )

    if diversify_titles:
        results_df = results_df.drop_duplicates(
            subset="project_title_key",
            keep="first",
        )

    results_df = (
        results_df.head(top_n)
        .reset_index(drop=True)
    )

    results_df.insert(
        0,
        "rank",
        range(1, len(results_df) + 1),
    )

    results_df["similarity_pct"] = (
        results_df["similarity_score"]
        * 100
    ).round(1)

    results_df["hybrid_score_pct"] = (
        results_df["hybrid_score"]
        * 100
    ).round(1)

    return results_df


hybrid_demo_results_df = hybrid_similarity_search(
    query=demo_query,
    top_n=10,
)

print(
    "Detected domain concepts:",
    detect_query_concepts(
        demo_query,
        DOMAIN_CONCEPT_LEXICON,
    ),
)

print(
    "Detected workflow concepts:",
    detect_query_concepts(
        demo_query,
        WORKFLOW_CONCEPT_LEXICON,
    ),
)

display(
    hybrid_demo_results_df[
        [
            "rank",
            "grant_key",
            "source",
            "title",
            "primary_topic",
            "similarity_pct",
            "domain_coverage",
            "workflow_coverage",
            "hybrid_score_pct",
        ]
    ]
)

Detected domain concepts: ['catalysis']
Detected workflow concepts: ['machine learning', 'automation', 'experimentation', 'closed loop', 'optimization', 'discovery']


,rank,grant_key,source,title,primary_topic,similarity_pct,domain_coverage,workflow_coverage,hybrid_score_pct
0,1,NSF_2231174,NSF,EAGER: ADAPT: Hypotheses Generation in Heterogeneous Catalysis using Causal Inference and Machine Learning,AI-enabled catalysis,11.2,1.0,0.333333,93.3
1,2,NSF_2306125,NSF,"Collaborative Research: DMREF: Machine Learning-aided Discovery of Synthesizable, Active and Stable Heterogeneous Ca...",AI-enabled catalysis,10.2,1.0,0.666667,90.0
2,3,NSF_2324157,NSF,Collaborative Research: DMREF: Computationally Driven Discovery and Synthesis of 2D Materials through Selective Etching,AI-enabled catalysis,7.5,1.0,0.833333,75.4
3,4,CORDIS_101105235,CORDIS,Computational Studies on Heterogeneous Astrocatalysis of Space-Abundant Transition Metals,AI-enabled catalysis,8.5,1.0,0.166667,74.9
4,5,NSF_2409631,NSF,Conference: Artificial Intelligence for Multidisciplinary Exploration and Discovery (AIMED) in Heterogeneous Catalys...,AI-enabled catalysis,7.7,1.0,0.666667,74.9
5,6,NSF_2203354,NSF,Emergence of Structure and Function from Sequenceable Sequence-Defined Macrocyclic Oligourethanes,AI-enabled catalysis,6.9,1.0,0.500000,68.3
6,7,NSF_2154428,NSF,Collaborative Research: A Data-driven Closed-loop Framework for De Novo Generation of Molecules with Targeted Proper...,AI-enabled catalysis,6.4,1.0,0.666667,66.3
7,8,NSF_2334969,NSF,Collaborative Research: Beyond the Single-Atom Paradigm: A Priori Design of Dual-Atom Alloy Active Sites for Efficie...,AI-enabled catalysis,6.4,1.0,0.500000,65.0
8,9,NSF_2339026,NSF,CAREER: Learning mechanistic models with automated experiments,Autonomous laboratories,9.3,0.0,0.666667,64.5
9,10,NSF_2323296,NSF,DMREF: Computationally-Driven Discovery of Designer 2D Materials for Biosensing,AI-enabled catalysis,5.7,1.0,0.833333,64.1


In [19]:
# Evaluate the hybrid model against the manual reference set

normalized_query = normalize_model_text(demo_query)

query_vector = improved_tfidf_vectorizer.transform(
    [normalized_query]
)

hybrid_similarity_scores = cosine_similarity(
    query_vector,
    improved_grant_tfidf_matrix,
).ravel()

hybrid_full_ranking_df = model_document_df.copy()

hybrid_full_ranking_df["similarity_score"] = (
    hybrid_similarity_scores
)

hybrid_full_ranking_df = hybrid_full_ranking_df.loc[
    hybrid_full_ranking_df["similarity_score"] > 0
].copy()

active_domain_concepts = detect_query_concepts(
    demo_query,
    DOMAIN_CONCEPT_LEXICON,
)

active_workflow_concepts = detect_query_concepts(
    demo_query,
    WORKFLOW_CONCEPT_LEXICON,
)

hybrid_full_ranking_df["tfidf_relative_score"] = (
    hybrid_full_ranking_df["similarity_score"]
    / hybrid_full_ranking_df["similarity_score"].max()
)

hybrid_full_ranking_df["domain_coverage"] = (
    hybrid_full_ranking_df["weighted_search_text"].apply(
        lambda text: calculate_concept_coverage(
            text,
            active_domain_concepts,
            DOMAIN_CONCEPT_LEXICON,
        )
    )
)

hybrid_full_ranking_df["workflow_coverage"] = (
    hybrid_full_ranking_df["weighted_search_text"].apply(
        lambda text: calculate_concept_coverage(
            text,
            active_workflow_concepts,
            WORKFLOW_CONCEPT_LEXICON,
        )
    )
)

# Use the same dynamic weights as the search function
if active_domain_concepts and active_workflow_concepts:
    tfidf_weight = 0.70
    domain_weight = 0.20
    workflow_weight = 0.10

elif active_domain_concepts:
    tfidf_weight = 0.80
    domain_weight = 0.20
    workflow_weight = 0.00

elif active_workflow_concepts:
    tfidf_weight = 0.85
    domain_weight = 0.00
    workflow_weight = 0.15

else:
    tfidf_weight = 1.00
    domain_weight = 0.00
    workflow_weight = 0.00

hybrid_full_ranking_df["hybrid_score"] = (
    tfidf_weight
    * hybrid_full_ranking_df["tfidf_relative_score"]
    + domain_weight
    * hybrid_full_ranking_df["domain_coverage"]
    + workflow_weight
    * hybrid_full_ranking_df["workflow_coverage"]
)

hybrid_full_ranking_df = (
    hybrid_full_ranking_df
    .sort_values(
        [
            "hybrid_score",
            "similarity_score",
            "award_year",
            "grant_key",
        ],
        ascending=[False, False, False, True],
    )
    .drop_duplicates(
        subset="project_title_key",
        keep="first",
    )
    .reset_index(drop=True)
)

hybrid_full_ranking_df["hybrid_rank"] = (
    hybrid_full_ranking_df.index + 1
)

hybrid_full_ranking_df["hybrid_score_pct"] = (
    hybrid_full_ranking_df["hybrid_score"]
    * 100
).round(1)

hybrid_ranking_lookup_df = hybrid_full_ranking_df[
    [
        "project_title_key",
        "hybrid_rank",
        "hybrid_score_pct",
        "domain_coverage",
        "workflow_coverage",
    ]
]

hybrid_evaluation_df = improved_evaluation_df.merge(
    hybrid_ranking_lookup_df,
    on="project_title_key",
    how="left",
    validate="many_to_one",
)

hybrid_evaluation_df["improvement_vs_improved_tfidf"] = (
    hybrid_evaluation_df["improved_tfidf_rank"]
    - hybrid_evaluation_df["hybrid_rank"]
)

hybrid_evaluation_df["hybrid_top_10"] = (
    hybrid_evaluation_df["hybrid_rank"] <= 10
)

hybrid_evaluation_df["hybrid_top_25"] = (
    hybrid_evaluation_df["hybrid_rank"] <= 25
)

hybrid_evaluation_df["hybrid_top_50"] = (
    hybrid_evaluation_df["hybrid_rank"] <= 50
)

hybrid_evaluation_df = hybrid_evaluation_df.sort_values(
    [
        "manual_relevance_score",
        "hybrid_rank",
    ],
    ascending=[False, True],
).reset_index(drop=True)

display(
    hybrid_evaluation_df[
        [
            "grant_key",
            "title",
            "manual_relevance_label",
            "improved_tfidf_rank",
            "hybrid_rank",
            "improvement_vs_improved_tfidf",
            "domain_coverage",
            "workflow_coverage",
            "hybrid_score_pct",
            "hybrid_top_10",
            "hybrid_top_25",
            "hybrid_top_50",
        ]
    ]
)

relevant_mask = (
    hybrid_evaluation_df["manual_relevance_score"] >= 2
)

hybrid_evaluation_summary_df = pd.DataFrame(
    {
        "metric": [
            "Relevant or strong projects",
            "Relevant or strong in top 10",
            "Relevant or strong in top 25",
            "Relevant or strong in top 50",
            "Strong matches in top 10",
            "Median improvement vs improved TF-IDF",
            "Weak-match hybrid rank",
        ],
        "value": [
            relevant_mask.sum(),
            (
                relevant_mask
                & hybrid_evaluation_df["hybrid_top_10"]
            ).sum(),
            (
                relevant_mask
                & hybrid_evaluation_df["hybrid_top_25"]
            ).sum(),
            (
                relevant_mask
                & hybrid_evaluation_df["hybrid_top_50"]
            ).sum(),
            (
                (
                    hybrid_evaluation_df[
                        "manual_relevance_score"
                    ] == 3
                )
                & hybrid_evaluation_df["hybrid_top_10"]
            ).sum(),
            hybrid_evaluation_df[
                "improvement_vs_improved_tfidf"
            ].median(),
            hybrid_evaluation_df.loc[
                hybrid_evaluation_df[
                    "manual_relevance_score"
                ] == 1,
                "hybrid_rank",
            ].min(),
        ],
    }
)

display(hybrid_evaluation_summary_df)

,grant_key,title,manual_relevance_label,improved_tfidf_rank,hybrid_rank,improvement_vs_improved_tfidf,domain_coverage,workflow_coverage,hybrid_score_pct,hybrid_top_10,hybrid_top_25,hybrid_top_50
0,NSF_2554343,Collaborative Research: DMREF: Atomically precise catalyst design for selective bond activation,Strong match,41,26,15,1.0,0.500000,52.4,False,False,True
1,CORDIS_101206634,Robot-mediated development of statistical models for mechanistic analysis amplification in synthetic organic reactions,Strong match,178,73,105,1.0,0.666667,41.3,False,False,False
2,NSF_2309852,Semi-Automated Discovery of Synthetic Polymers with Protein Features,Relevant,24,14,10,1.0,0.666667,61.5,False,True,True
3,CORDIS_101062692,Computationally driven discovery of organic dyes for photoredox catalysis from physicochemical principles and mechan...,Relevant,99,38,61,1.0,0.833333,46.8,False,False,True
4,CORDIS_101118768,Directed Evolution of Metastable Electrocatalyst Interfaces for Energy Conversion,Relevant,93,42,51,1.0,0.666667,45.5,False,False,True
5,NSF_2318141,CCI Phase I: NSF Center for Sustainable Photoredox Catalysis (SuPRCat),Relevant,79,43,36,1.0,0.500000,45.2,False,False,True
6,CORDIS_101204747,Development of Data-assisted Photo-Organocatalytic Transformations,Relevant,371,156,215,1.0,0.500000,34.8,False,False,False
7,CORDIS_101098001,"Automated, miniaturized and accelerated drug discovery: AMADEUS",Weak match,54,27,27,1.0,0.666667,51.3,False,False,True


,metric,value
0,Relevant or strong projects,7.0
1,Relevant or strong in top 10,0.0
2,Relevant or strong in top 25,1.0
3,Relevant or strong in top 50,5.0
4,Strong matches in top 10,0.0
5,Median improvement vs improved TF-IDF,43.5
6,Weak-match hybrid rank,27.0


In [20]:
# Diagnose where domain and workflow concept matches occur

def find_matching_terms(
    text,
    active_concepts,
    concept_lexicon,
):
    """Return the exact controlled terms found in a text field."""

    matched_terms = []

    for concept_name in active_concepts:
        for term in concept_lexicon[concept_name]:
            if contains_term(text, term):
                matched_terms.append(term)

    return sorted(set(matched_terms))


def find_matching_concepts(
    text,
    active_concepts,
    concept_lexicon,
):
    """Return the controlled concepts represented in a text field."""

    matched_concepts = []

    for concept_name in active_concepts:
        if any(
            contains_term(text, term)
            for term in concept_lexicon[concept_name]
        ):
            matched_concepts.append(concept_name)

    return matched_concepts


# Use the representative document selected for each title family
representative_lookup_df = (
    hybrid_full_ranking_df
    .set_index("project_title_key")
)

hybrid_diagnostic_df = hybrid_evaluation_df.copy()

hybrid_diagnostic_df["normalized_title"] = (
    hybrid_diagnostic_df["project_title_key"]
    .map(
        representative_lookup_df[
            "normalized_title"
        ]
    )
)

hybrid_diagnostic_df["normalized_abstract"] = (
    hybrid_diagnostic_df["project_title_key"]
    .map(
        representative_lookup_df[
            "normalized_abstract"
        ]
    )
)

hybrid_diagnostic_df["title_domain_terms"] = (
    hybrid_diagnostic_df["normalized_title"]
    .apply(
        lambda text: find_matching_terms(
            text,
            active_domain_concepts,
            DOMAIN_CONCEPT_LEXICON,
        )
    )
)

hybrid_diagnostic_df["abstract_domain_terms"] = (
    hybrid_diagnostic_df["normalized_abstract"]
    .apply(
        lambda text: find_matching_terms(
            text,
            active_domain_concepts,
            DOMAIN_CONCEPT_LEXICON,
        )
    )
)

hybrid_diagnostic_df["title_workflow_concepts"] = (
    hybrid_diagnostic_df["normalized_title"]
    .apply(
        lambda text: find_matching_concepts(
            text,
            active_workflow_concepts,
            WORKFLOW_CONCEPT_LEXICON,
        )
    )
)

hybrid_diagnostic_df["abstract_workflow_concepts"] = (
    hybrid_diagnostic_df["normalized_abstract"]
    .apply(
        lambda text: find_matching_concepts(
            text,
            active_workflow_concepts,
            WORKFLOW_CONCEPT_LEXICON,
        )
    )
)

display(
    hybrid_diagnostic_df[
        [
            "grant_key",
            "title",
            "manual_relevance_label",
            "hybrid_rank",
            "title_domain_terms",
            "abstract_domain_terms",
            "title_workflow_concepts",
            "abstract_workflow_concepts",
        ]
    ]
)

,grant_key,title,manual_relevance_label,hybrid_rank,title_domain_terms,abstract_domain_terms,title_workflow_concepts,abstract_workflow_concepts
0,NSF_2554343,Collaborative Research: DMREF: Atomically precise catalyst design for selective bond activation,Strong match,26,[catalyst],[catalyst],[discovery],"[machine learning, experimentation, discovery]"
1,CORDIS_101206634,Robot-mediated development of statistical models for mechanistic analysis amplification in synthetic organic reactions,Strong match,73,[],[catalyst],[automation],"[machine learning, automation, optimization, discovery]"
2,NSF_2309852,Semi-Automated Discovery of Synthetic Polymers with Protein Features,Relevant,14,[],[catalyst],"[automation, discovery]","[machine learning, automation, closed loop, discovery]"
3,CORDIS_101062692,Computationally driven discovery of organic dyes for photoredox catalysis from physicochemical principles and mechan...,Relevant,38,[catalyst],[catalyst],[discovery],"[machine learning, automation, experimentation, optimization, discovery]"
4,CORDIS_101118768,Directed Evolution of Metastable Electrocatalyst Interfaces for Energy Conversion,Relevant,42,[electrocatalyst],[catalyst],[],"[machine learning, experimentation, optimization, discovery]"
5,NSF_2318141,CCI Phase I: NSF Center for Sustainable Photoredox Catalysis (SuPRCat),Relevant,43,[catalyst],[catalyst],[],"[machine learning, optimization, discovery]"
6,CORDIS_101204747,Development of Data-assisted Photo-Organocatalytic Transformations,Relevant,156,[],"[catalyst, photocatalyst]",[],"[machine learning, experimentation, discovery]"
7,CORDIS_101098001,"Automated, miniaturized and accelerated drug discovery: AMADEUS",Weak match,27,[],[catalyst],"[automation, discovery]","[machine learning, automation, optimization, discovery]"


In [21]:
# Build a field-aware hybrid ranking

def binary_concept_match(
    text,
    active_concepts,
    concept_lexicon,
):
    """Return 1 when any active concept appears in the text."""

    if not active_concepts:
        return 0.0

    matched_concepts = find_matching_concepts(
        text,
        active_concepts,
        concept_lexicon,
    )

    return float(len(matched_concepts) > 0)


field_aware_full_ranking_df = model_document_df.copy()

# Reuse the improved TF-IDF similarity scores
field_aware_full_ranking_df["similarity_score"] = (
    cosine_similarity(
        improved_tfidf_vectorizer.transform(
            [normalize_model_text(demo_query)]
        ),
        improved_grant_tfidf_matrix,
    ).ravel()
)

field_aware_full_ranking_df = (
    field_aware_full_ranking_df.loc[
        field_aware_full_ranking_df[
            "similarity_score"
        ] > 0
    ].copy()
)

# Relative TF-IDF score
field_aware_full_ranking_df[
    "tfidf_relative_score"
] = (
    field_aware_full_ranking_df[
        "similarity_score"
    ]
    / field_aware_full_ranking_df[
        "similarity_score"
    ].max()
)

# Domain evidence by text field
field_aware_full_ranking_df[
    "title_domain_match"
] = (
    field_aware_full_ranking_df[
        "normalized_title"
    ].apply(
        lambda text: binary_concept_match(
            text,
            active_domain_concepts,
            DOMAIN_CONCEPT_LEXICON,
        )
    )
)

field_aware_full_ranking_df[
    "abstract_domain_match"
] = (
    field_aware_full_ranking_df[
        "normalized_abstract"
    ].apply(
        lambda text: binary_concept_match(
            text,
            active_domain_concepts,
            DOMAIN_CONCEPT_LEXICON,
        )
    )
)

# Title evidence receives full credit.
# Abstract-only evidence receives partial credit.
field_aware_full_ranking_df[
    "domain_centrality"
] = np.where(
    field_aware_full_ranking_df[
        "title_domain_match"
    ] == 1,
    1.0,
    np.where(
        field_aware_full_ranking_df[
            "abstract_domain_match"
        ] == 1,
        0.35,
        0.0,
    ),
)

# Workflow coverage by text field
field_aware_full_ranking_df[
    "title_workflow_coverage"
] = (
    field_aware_full_ranking_df[
        "normalized_title"
    ].apply(
        lambda text: calculate_concept_coverage(
            text,
            active_workflow_concepts,
            WORKFLOW_CONCEPT_LEXICON,
        )
    )
)

field_aware_full_ranking_df[
    "abstract_workflow_coverage"
] = (
    field_aware_full_ranking_df[
        "normalized_abstract"
    ].apply(
        lambda text: calculate_concept_coverage(
            text,
            active_workflow_concepts,
            WORKFLOW_CONCEPT_LEXICON,
        )
    )
)

field_aware_full_ranking_df[
    "workflow_centrality"
] = (
    0.60
    * field_aware_full_ranking_df[
        "title_workflow_coverage"
    ]
    + 0.40
    * field_aware_full_ranking_df[
        "abstract_workflow_coverage"
    ]
)

# Final explainable score
field_aware_full_ranking_df[
    "field_aware_score"
] = (
    0.65
    * field_aware_full_ranking_df[
        "tfidf_relative_score"
    ]
    + 0.25
    * field_aware_full_ranking_df[
        "domain_centrality"
    ]
    + 0.10
    * field_aware_full_ranking_df[
        "workflow_centrality"
    ]
)

field_aware_full_ranking_df = (
    field_aware_full_ranking_df
    .sort_values(
        [
            "field_aware_score",
            "similarity_score",
            "award_year",
            "grant_key",
        ],
        ascending=[False, False, False, True],
    )
    .drop_duplicates(
        subset="project_title_key",
        keep="first",
    )
    .reset_index(drop=True)
)

field_aware_full_ranking_df[
    "field_aware_rank"
] = (
    field_aware_full_ranking_df.index + 1
)

field_aware_full_ranking_df[
    "field_aware_score_pct"
] = (
    field_aware_full_ranking_df[
        "field_aware_score"
    ]
    * 100
).round(1)

# Attach the new ranking to the benchmark
field_aware_lookup_df = (
    field_aware_full_ranking_df[
        [
            "project_title_key",
            "field_aware_rank",
            "field_aware_score_pct",
            "title_domain_match",
            "abstract_domain_match",
            "domain_centrality",
            "workflow_centrality",
        ]
    ]
)

field_aware_evaluation_df = (
    hybrid_evaluation_df.merge(
        field_aware_lookup_df,
        on="project_title_key",
        how="left",
        validate="many_to_one",
    )
)

field_aware_evaluation_df[
    "improvement_vs_hybrid"
] = (
    field_aware_evaluation_df[
        "hybrid_rank"
    ]
    - field_aware_evaluation_df[
        "field_aware_rank"
    ]
)

field_aware_evaluation_df = (
    field_aware_evaluation_df
    .sort_values(
        [
            "manual_relevance_score",
            "field_aware_rank",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(
    field_aware_evaluation_df[
        [
            "grant_key",
            "title",
            "manual_relevance_label",
            "hybrid_rank",
            "field_aware_rank",
            "improvement_vs_hybrid",
            "title_domain_match",
            "abstract_domain_match",
            "domain_centrality",
            "workflow_centrality",
            "field_aware_score_pct",
        ]
    ]
)

display(
    field_aware_full_ranking_df[
        [
            "field_aware_rank",
            "grant_key",
            "source",
            "title",
            "primary_topic",
            "domain_centrality",
            "workflow_centrality",
            "field_aware_score_pct",
        ]
    ].head(10)
)

,grant_key,title,manual_relevance_label,hybrid_rank,field_aware_rank,improvement_vs_hybrid,title_domain_match,abstract_domain_match,domain_centrality,workflow_centrality,field_aware_score_pct
0,NSF_2554343,Collaborative Research: DMREF: Atomically precise catalyst design for selective bond activation,Strong match,26,15,11,1.0,1.0,1.00,0.300000,53.5
1,CORDIS_101206634,Robot-mediated development of statistical models for mechanistic analysis amplification in synthetic organic reactions,Strong match,73,134,-61,0.0,1.0,0.35,0.366667,26.0
2,CORDIS_101062692,Computationally driven discovery of organic dyes for photoredox catalysis from physicochemical principles and mechan...,Relevant,38,28,10,1.0,1.0,1.00,0.433333,46.5
3,NSF_2309852,Semi-Automated Discovery of Synthetic Polymers with Protein Features,Relevant,14,31,-17,0.0,1.0,0.35,0.466667,45.8
4,NSF_2318141,CCI Phase I: NSF Center for Sustainable Photoredox Catalysis (SuPRCat),Relevant,43,32,11,1.0,1.0,1.00,0.200000,45.8
5,CORDIS_101118768,Directed Evolution of Metastable Electrocatalyst Interfaces for Energy Conversion,Relevant,42,35,7,1.0,1.0,1.00,0.266667,45.2
6,CORDIS_101204747,Development of Data-assisted Photo-Organocatalytic Transformations,Relevant,156,209,-53,0.0,1.0,0.35,0.200000,19.9
7,CORDIS_101098001,"Automated, miniaturized and accelerated drug discovery: AMADEUS",Weak match,27,87,-60,0.0,1.0,0.35,0.466667,36.3


,field_aware_rank,grant_key,source,title,primary_topic,domain_centrality,workflow_centrality,field_aware_score_pct
0,1,NSF_2231174,NSF,EAGER: ADAPT: Hypotheses Generation in Heterogeneous Catalysis using Causal Inference and Machine Learning,AI-enabled catalysis,1.00,0.233333,92.3
1,2,NSF_2306125,NSF,"Collaborative Research: DMREF: Machine Learning-aided Discovery of Synthesizable, Active and Stable Heterogeneous Ca...",AI-enabled catalysis,1.00,0.466667,88.5
2,3,NSF_2409631,NSF,Conference: Artificial Intelligence for Multidisciplinary Exploration and Discovery (AIMED) in Heterogeneous Catalys...,AI-enabled catalysis,1.00,0.466667,74.4
3,4,NSF_2420839,NSF,Bimetallic Single-Site Heterogeneous Catalysts: A New Paradigm for Enhancing Reactivity in Olefin Metathesis,AI-enabled catalysis,1.00,0.200000,61.5
4,5,CORDIS_101168623,CORDIS,Closing the loop in stereoselective catalysis with data-driven approaches,AI-enabled catalysis,1.00,0.233333,61.1
5,6,NSF_2143941,NSF,CAREER: Understanding metal/support interactions in catalysis with statistical learning,AI-enabled catalysis,1.00,0.266667,59.4
6,7,NSF_2235778,NSF,CAREER: Advancing Light-mediated Ni Catalysis using Data Science and Physical Organic Techniques,Reaction prediction,1.00,0.066667,59.2
7,8,CORDIS_101105235,CORDIS,Computational Studies on Heterogeneous Astrocatalysis of Space-Abundant Transition Metals,AI-enabled catalysis,0.35,0.066667,58.8
8,9,NSF_2154237,NSF,Discovery and Optimization of Enantioselective Catalysts Guided by Informatics and Machine Learning,AI-enabled catalysis,1.00,0.500000,58.5
9,10,NSF_2339026,NSF,CAREER: Learning mechanistic models with automated experiments,Autonomous laboratories,0.00,0.466667,58.4


In [22]:
# Measure how strongly the query domain appears in each benchmark abstract

def count_matching_term_occurrences(
    text,
    active_concepts,
    concept_lexicon,
):
    """Count occurrences of controlled domain terms in normalized text."""

    text = str(text)

    term_counts = {}

    for concept_name in active_concepts:
        for term in concept_lexicon[concept_name]:
            count = len(
                re.findall(
                    rf"\b{re.escape(term)}\b",
                    text,
                )
            )

            if count > 0:
                term_counts[term] = count

    return term_counts


def total_matching_occurrences(term_counts):
    """Sum occurrences stored in a term-count dictionary."""

    return sum(term_counts.values())


domain_frequency_diagnostic_df = (
    field_aware_evaluation_df.copy()
)

domain_frequency_diagnostic_df[
    "normalized_abstract"
] = (
    domain_frequency_diagnostic_df[
        "project_title_key"
    ].map(
        representative_lookup_df[
            "normalized_abstract"
        ]
    )
)

domain_frequency_diagnostic_df[
    "abstract_domain_term_counts"
] = (
    domain_frequency_diagnostic_df[
        "normalized_abstract"
    ].apply(
        lambda text: count_matching_term_occurrences(
            text,
            active_domain_concepts,
            DOMAIN_CONCEPT_LEXICON,
        )
    )
)

domain_frequency_diagnostic_df[
    "abstract_domain_occurrences"
] = (
    domain_frequency_diagnostic_df[
        "abstract_domain_term_counts"
    ].apply(
        total_matching_occurrences
    )
)

domain_frequency_diagnostic_df[
    "abstract_word_count"
] = (
    domain_frequency_diagnostic_df[
        "normalized_abstract"
    ].str.split().str.len()
)

domain_frequency_diagnostic_df[
    "domain_mentions_per_100_words"
] = (
    domain_frequency_diagnostic_df[
        "abstract_domain_occurrences"
    ]
    / domain_frequency_diagnostic_df[
        "abstract_word_count"
    ]
    * 100
).round(2)

display(
    domain_frequency_diagnostic_df[
        [
            "grant_key",
            "title",
            "manual_relevance_label",
            "field_aware_rank",
            "abstract_domain_term_counts",
            "abstract_domain_occurrences",
            "abstract_word_count",
            "domain_mentions_per_100_words",
        ]
    ]
)

,grant_key,title,manual_relevance_label,field_aware_rank,abstract_domain_term_counts,abstract_domain_occurrences,abstract_word_count,domain_mentions_per_100_words
0,NSF_2554343,Collaborative Research: DMREF: Atomically precise catalyst design for selective bond activation,Strong match,15,{'catalyst': 8},8,472,1.69
1,CORDIS_101206634,Robot-mediated development of statistical models for mechanistic analysis amplification in synthetic organic reactions,Strong match,134,{'catalyst': 1},1,146,0.68
2,CORDIS_101062692,Computationally driven discovery of organic dyes for photoredox catalysis from physicochemical principles and mechan...,Relevant,28,{'catalyst': 2},2,291,0.69
3,NSF_2309852,Semi-Automated Discovery of Synthetic Polymers with Protein Features,Relevant,31,{'catalyst': 4},4,364,1.10
4,NSF_2318141,CCI Phase I: NSF Center for Sustainable Photoredox Catalysis (SuPRCat),Relevant,32,{'catalyst': 11},11,427,2.58
5,CORDIS_101118768,Directed Evolution of Metastable Electrocatalyst Interfaces for Energy Conversion,Relevant,35,{'catalyst': 2},2,259,0.77
6,CORDIS_101204747,Development of Data-assisted Photo-Organocatalytic Transformations,Relevant,209,"{'catalyst': 2, 'photocatalyst': 2}",4,266,1.50
7,CORDIS_101098001,"Automated, miniaturized and accelerated drug discovery: AMADEUS",Weak match,87,{'catalyst': 1},1,281,0.36


In [23]:
# Test a context-aware ranking that distinguishes central domain
# relevance from incidental abstract mentions

CONTEXT_CONCEPT_LEXICON = {
    "chemistry": (
        "chemical",
        "chemistry",
        "synthesis",
        "synthetic",
        "organic",
        "molecule",
        "molecular",
    ),
    "reactions": (
        "reaction",
        "reactivity",
        "mechanism",
        "mechanistic",
        "bond activation",
        "transformation",
    ),
    "materials": (
        "material",
        "alloy",
        "polymer",
        "semiconductor",
        "framework",
        "composite",
    ),
}

DOMAIN_CONTEXT_EXPANSION = {
    "catalysis": ("chemistry", "reactions"),
    "materials": ("materials", "chemistry"),
    "chemistry": ("chemistry", "reactions"),
    "reactions": ("reactions", "chemistry"),
}

APPLICATION_LEXICON = {
    "drug discovery": (
        "drug discovery",
        "drug",
        "therapeutic",
        "pharmaceutical",
    ),
    "biology": (
        "protein",
        "enzyme",
        "biological",
        "biofoundry",
    ),
}

APPLICATION_PENALTIES = {
    "drug discovery": 1.00,
    "biology": 0.35,
}


def abstract_domain_density_score(text):
    """Return capped domain mentions per 100 abstract words."""

    term_counts = count_matching_term_occurrences(
        text,
        active_domain_concepts,
        DOMAIN_CONCEPT_LEXICON,
    )

    occurrence_count = sum(term_counts.values())
    word_count = max(len(str(text).split()), 1)

    mentions_per_100_words = (
        occurrence_count / word_count * 100
    )

    return min(mentions_per_100_words / 1.0, 1.0)


def calculate_off_domain_penalty(title, query):
    """Penalize unrelated application areas expressed in the title."""

    active_applications = detect_query_concepts(
        query,
        APPLICATION_LEXICON,
    )

    title_applications = find_matching_concepts(
        title,
        list(APPLICATION_LEXICON.keys()),
        APPLICATION_LEXICON,
    )

    inactive_title_applications = [
        application
        for application in title_applications
        if application not in active_applications
    ]

    return min(
        sum(
            APPLICATION_PENALTIES[application]
            for application in inactive_title_applications
        ),
        1.0,
    )


# Expand the specific query domain into broader context concepts
active_context_concepts = sorted(
    {
        context_concept
        for domain_concept in active_domain_concepts
        for context_concept in DOMAIN_CONTEXT_EXPANSION.get(
            domain_concept,
            (),
        )
    }
)

context_aware_full_ranking_df = model_document_df.copy()

context_aware_full_ranking_df["similarity_score"] = (
    cosine_similarity(
        improved_tfidf_vectorizer.transform(
            [normalize_model_text(demo_query)]
        ),
        improved_grant_tfidf_matrix,
    ).ravel()
)

context_aware_full_ranking_df = (
    context_aware_full_ranking_df.loc[
        context_aware_full_ranking_df["similarity_score"] > 0
    ].copy()
)

context_aware_full_ranking_df["tfidf_relative_score"] = (
    context_aware_full_ranking_df["similarity_score"]
    / context_aware_full_ranking_df["similarity_score"].max()
)

# Specific query-domain evidence
context_aware_full_ranking_df["title_domain_match"] = (
    context_aware_full_ranking_df["normalized_title"].apply(
        lambda text: binary_concept_match(
            text,
            active_domain_concepts,
            DOMAIN_CONCEPT_LEXICON,
        )
    )
)

context_aware_full_ranking_df["abstract_domain_density"] = (
    context_aware_full_ranking_df["normalized_abstract"].apply(
        abstract_domain_density_score
    )
)

context_aware_full_ranking_df["specific_domain_score"] = np.where(
    context_aware_full_ranking_df["title_domain_match"] == 1,
    1.0,
    0.50
    * context_aware_full_ranking_df["abstract_domain_density"],
)

# Broader chemistry/reaction context
context_aware_full_ranking_df["title_context_coverage"] = (
    context_aware_full_ranking_df["normalized_title"].apply(
        lambda text: calculate_concept_coverage(
            text,
            active_context_concepts,
            CONTEXT_CONCEPT_LEXICON,
        )
    )
)

context_aware_full_ranking_df["abstract_context_coverage"] = (
    context_aware_full_ranking_df["normalized_abstract"].apply(
        lambda text: calculate_concept_coverage(
            text,
            active_context_concepts,
            CONTEXT_CONCEPT_LEXICON,
        )
    )
)

context_aware_full_ranking_df["context_score"] = (
    0.75
    * context_aware_full_ranking_df["title_context_coverage"]
    + 0.25
    * context_aware_full_ranking_df["abstract_context_coverage"]
)

# Research-workflow coverage
context_aware_full_ranking_df["workflow_score"] = (
    0.60
    * context_aware_full_ranking_df["normalized_title"].apply(
        lambda text: calculate_concept_coverage(
            text,
            active_workflow_concepts,
            WORKFLOW_CONCEPT_LEXICON,
        )
    )
    + 0.40
    * context_aware_full_ranking_df["normalized_abstract"].apply(
        lambda text: calculate_concept_coverage(
            text,
            active_workflow_concepts,
            WORKFLOW_CONCEPT_LEXICON,
        )
    )
)

# Small penalty for an unrelated application stated in the title
context_aware_full_ranking_df["off_domain_penalty"] = (
    context_aware_full_ranking_df["normalized_title"].apply(
        lambda title: calculate_off_domain_penalty(
            title,
            demo_query,
        )
    )
)

context_aware_full_ranking_df["context_aware_score"] = (
    0.58
    * context_aware_full_ranking_df["tfidf_relative_score"]
    + 0.17
    * context_aware_full_ranking_df["specific_domain_score"]
    + 0.12
    * context_aware_full_ranking_df["context_score"]
    + 0.13
    * context_aware_full_ranking_df["workflow_score"]
    - 0.10
    * context_aware_full_ranking_df["off_domain_penalty"]
)

context_aware_full_ranking_df = (
    context_aware_full_ranking_df.sort_values(
        [
            "context_aware_score",
            "similarity_score",
            "award_year",
            "grant_key",
        ],
        ascending=[False, False, False, True],
    )
    .drop_duplicates(
        subset="project_title_key",
        keep="first",
    )
    .reset_index(drop=True)
)

context_aware_full_ranking_df["context_aware_rank"] = (
    context_aware_full_ranking_df.index + 1
)

context_lookup_df = context_aware_full_ranking_df[
    [
        "project_title_key",
        "context_aware_rank",
        "specific_domain_score",
        "context_score",
        "workflow_score",
        "off_domain_penalty",
        "context_aware_score",
    ]
]

context_evaluation_df = field_aware_evaluation_df.merge(
    context_lookup_df,
    on="project_title_key",
    how="left",
    validate="many_to_one",
)

context_evaluation_df["improvement_vs_field_aware"] = (
    context_evaluation_df["field_aware_rank"]
    - context_evaluation_df["context_aware_rank"]
)

context_evaluation_df = context_evaluation_df.sort_values(
    ["manual_relevance_score", "context_aware_rank"],
    ascending=[False, True],
).reset_index(drop=True)

display(
    context_evaluation_df[
        [
            "grant_key",
            "title",
            "manual_relevance_label",
            "field_aware_rank",
            "context_aware_rank",
            "improvement_vs_field_aware",
            "specific_domain_score",
            "context_score",
            "workflow_score",
            "off_domain_penalty",
        ]
    ]
)

,grant_key,title,manual_relevance_label,field_aware_rank,context_aware_rank,improvement_vs_field_aware,specific_domain_score,context_score,workflow_score,off_domain_penalty
0,NSF_2554343,Collaborative Research: DMREF: Atomically precise catalyst design for selective bond activation,Strong match,15,14,1,1.000000,0.625,0.300000,0.00
1,CORDIS_101206634,Robot-mediated development of statistical models for mechanistic analysis amplification in synthetic organic reactions,Strong match,134,75,59,0.342466,1.000,0.366667,0.00
2,CORDIS_101062692,Computationally driven discovery of organic dyes for photoredox catalysis from physicochemical principles and mechan...,Relevant,28,17,11,1.000000,1.000,0.433333,0.00
3,NSF_2309852,Semi-Automated Discovery of Synthetic Polymers with Protein Features,Relevant,31,20,11,0.500000,0.500,0.466667,0.35
4,NSF_2318141,CCI Phase I: NSF Center for Sustainable Photoredox Catalysis (SuPRCat),Relevant,32,39,-7,1.000000,0.250,0.200000,0.00
5,CORDIS_101118768,Directed Evolution of Metastable Electrocatalyst Interfaces for Energy Conversion,Relevant,35,42,-7,1.000000,0.250,0.266667,0.00
6,CORDIS_101204747,Development of Data-assisted Photo-Organocatalytic Transformations,Relevant,209,184,25,0.500000,0.250,0.200000,0.00
7,CORDIS_101098001,"Automated, miniaturized and accelerated drug discovery: AMADEUS",Weak match,87,212,-125,0.177936,0.125,0.466667,1.00


### Final ranking-model selection

The context-aware hybrid model was selected for the final search tool.

It combines:
- normalized, title-weighted TF-IDF similarity;
- specific scientific-domain evidence;
- broader chemistry and reaction context;
- research-workflow coverage;
- a small penalty for clearly unrelated application areas.

Compared with earlier versions, it substantially lowered the weak drug-discovery result while preserving or improving the ranking of most chemistry- and catalyst-focused projects.

The eight-project benchmark is small and query-specific, so it is treated as a qualitative model check rather than a formal accuracy estimate. Further tuning against these examples was avoided to reduce overfitting.

In [24]:
# Create the final reusable context-aware search function

def context_aware_similarity_search(
    query,
    top_n=10,
    source=None,
    topic=None,
    relevance_tier=None,
    year_min=None,
    year_max=None,
    diversify_titles=True,
):
    """Return explainable grant recommendations for a research concept."""

    if not str(query).strip():
        raise ValueError(
            "The research concept cannot be empty."
        )

    normalized_query = normalize_model_text(query)

    query_vector = improved_tfidf_vectorizer.transform(
        [normalized_query]
    )

    if query_vector.nnz == 0:
        raise ValueError(
            "The query contains no terms recognized by the model."
        )

    similarity_scores = cosine_similarity(
        query_vector,
        improved_grant_tfidf_matrix,
    ).ravel()

    results_df = model_document_df.copy()

    results_df["similarity_score"] = similarity_scores

    # Apply optional metadata filters
    if source is not None:
        results_df = results_df.loc[
            results_df["source"] == source
        ].copy()

    if topic is not None:
        results_df = results_df.loc[
            results_df["primary_topic"] == topic
        ].copy()

    if relevance_tier is not None:
        results_df = results_df.loc[
            results_df["relevance_tier"] == relevance_tier
        ].copy()

    if year_min is not None:
        results_df = results_df.loc[
            results_df["award_year"] >= year_min
        ].copy()

    if year_max is not None:
        results_df = results_df.loc[
            results_df["award_year"] <= year_max
        ].copy()

    results_df = results_df.loc[
        results_df["similarity_score"] > 0
    ].copy()

    if results_df.empty:
        return results_df

    # Detect concepts represented in the current query
    query_domain_concepts = detect_query_concepts(
        query,
        DOMAIN_CONCEPT_LEXICON,
    )

    query_workflow_concepts = detect_query_concepts(
        query,
        WORKFLOW_CONCEPT_LEXICON,
    )

    query_context_concepts = sorted(
        {
            context_concept
            for domain_concept in query_domain_concepts
            for context_concept in DOMAIN_CONTEXT_EXPANSION.get(
                domain_concept,
                (),
            )
        }
    )

    results_df["tfidf_relative_score"] = (
        results_df["similarity_score"]
        / results_df["similarity_score"].max()
    )

    # Specific domain evidence
    if query_domain_concepts:
        results_df["title_domain_match"] = (
            results_df["normalized_title"].apply(
                lambda text: binary_concept_match(
                    text,
                    query_domain_concepts,
                    DOMAIN_CONCEPT_LEXICON,
                )
            )
        )

        results_df["abstract_domain_density"] = (
            results_df["normalized_abstract"].apply(
                lambda text: min(
                    (
                        sum(
                            count_matching_term_occurrences(
                                text,
                                query_domain_concepts,
                                DOMAIN_CONCEPT_LEXICON,
                            ).values()
                        )
                        / max(len(str(text).split()), 1)
                        * 100
                    ),
                    1.0,
                )
            )
        )

        results_df["specific_domain_score"] = np.where(
            results_df["title_domain_match"] == 1,
            1.0,
            0.50 * results_df["abstract_domain_density"],
        )

    else:
        results_df["title_domain_match"] = 0.0
        results_df["abstract_domain_density"] = 0.0
        results_df["specific_domain_score"] = 0.0

    # Broader scientific context
    if query_context_concepts:
        results_df["context_score"] = (
            0.75
            * results_df["normalized_title"].apply(
                lambda text: calculate_concept_coverage(
                    text,
                    query_context_concepts,
                    CONTEXT_CONCEPT_LEXICON,
                )
            )
            + 0.25
            * results_df["normalized_abstract"].apply(
                lambda text: calculate_concept_coverage(
                    text,
                    query_context_concepts,
                    CONTEXT_CONCEPT_LEXICON,
                )
            )
        )

    else:
        results_df["context_score"] = 0.0

    # Research-workflow evidence
    if query_workflow_concepts:
        results_df["workflow_score"] = (
            0.60
            * results_df["normalized_title"].apply(
                lambda text: calculate_concept_coverage(
                    text,
                    query_workflow_concepts,
                    WORKFLOW_CONCEPT_LEXICON,
                )
            )
            + 0.40
            * results_df["normalized_abstract"].apply(
                lambda text: calculate_concept_coverage(
                    text,
                    query_workflow_concepts,
                    WORKFLOW_CONCEPT_LEXICON,
                )
            )
        )

    else:
        results_df["workflow_score"] = 0.0

    results_df["off_domain_penalty"] = (
        results_df["normalized_title"].apply(
            lambda title: calculate_off_domain_penalty(
                title,
                query,
            )
        )
    )

    # Redistribute unavailable concept weights to TF-IDF
    tfidf_weight = 0.58
    domain_weight = 0.17
    context_weight = 0.12
    workflow_weight = 0.13

    if not query_domain_concepts:
        tfidf_weight += domain_weight
        domain_weight = 0.0

    if not query_context_concepts:
        tfidf_weight += context_weight
        context_weight = 0.0

    if not query_workflow_concepts:
        tfidf_weight += workflow_weight
        workflow_weight = 0.0

    results_df["final_score"] = (
        tfidf_weight
        * results_df["tfidf_relative_score"]
        + domain_weight
        * results_df["specific_domain_score"]
        + context_weight
        * results_df["context_score"]
        + workflow_weight
        * results_df["workflow_score"]
        - 0.10
        * results_df["off_domain_penalty"]
    )

    results_df = results_df.sort_values(
        [
            "final_score",
            "similarity_score",
            "award_year",
            "grant_key",
        ],
        ascending=[False, False, False, True],
    )

    if diversify_titles:
        results_df = results_df.drop_duplicates(
            subset="project_title_key",
            keep="first",
        )

    results_df = (
        results_df.head(top_n)
        .reset_index(drop=True)
    )

    results_df.insert(
        0,
        "rank",
        range(1, len(results_df) + 1),
    )

    results_df["similarity_pct"] = (
        results_df["similarity_score"] * 100
    ).round(1)

    results_df["final_score_pct"] = (
        results_df["final_score"] * 100
    ).round(1)

    results_df["detected_domain_concepts"] = [
        query_domain_concepts
    ] * len(results_df)

    results_df["detected_workflow_concepts"] = [
        query_workflow_concepts
    ] * len(results_df)

    return results_df


# Test the final reusable function
final_demo_results_df = context_aware_similarity_search(
    query=demo_query,
    top_n=10,
)

display(
    final_demo_results_df[
        [
            "rank",
            "grant_key",
            "source",
            "title",
            "primary_topic",
            "award_year",
            "similarity_pct",
            "specific_domain_score",
            "context_score",
            "workflow_score",
            "off_domain_penalty",
            "final_score_pct",
        ]
    ]
)

print("Results returned:", len(final_demo_results_df))

print(
    "Duplicate title families:",
    final_demo_results_df[
        "project_title_key"
    ].duplicated().sum(),
)

,rank,grant_key,source,title,primary_topic,award_year,similarity_pct,specific_domain_score,context_score,workflow_score,off_domain_penalty,final_score_pct
0,1,NSF_2231174,NSF,EAGER: ADAPT: Hypotheses Generation in Heterogeneous Catalysis using Causal Inference and Machine Learning,AI-enabled catalysis,2022,11.2,1.00000,0.250,0.233333,0.0,81.0
1,2,NSF_2306125,NSF,"Collaborative Research: DMREF: Machine Learning-aided Discovery of Synthesizable, Active and Stable Heterogeneous Ca...",AI-enabled catalysis,2022,10.2,1.00000,0.250,0.466667,0.0,78.5
2,3,NSF_2409631,NSF,Conference: Artificial Intelligence for Multidisciplinary Exploration and Discovery (AIMED) in Heterogeneous Catalys...,AI-enabled catalysis,2024,7.7,1.00000,0.125,0.466667,0.0,64.5
3,4,NSF_2339026,NSF,CAREER: Learning mechanistic models with automated experiments,Autonomous laboratories,2024,9.3,0.00000,0.625,0.466667,0.0,61.5
4,5,NSF_2420839,NSF,Bimetallic Single-Site Heterogeneous Catalysts: A New Paradigm for Enhancing Reactivity in Olefin Metathesis,AI-enabled catalysis,2024,6.0,1.00000,0.625,0.200000,0.0,57.8
5,6,CORDIS_101105235,CORDIS,Computational Studies on Heterogeneous Astrocatalysis of Space-Abundant Transition Metals,AI-enabled catalysis,2024,8.5,0.50000,0.250,0.066667,0.0,56.5
6,7,NSF_2235778,NSF,CAREER: Advancing Light-mediated Ni Catalysis using Data Science and Physical Organic Techniques,Reaction prediction,2023,5.8,1.00000,0.625,0.066667,0.0,55.3
7,8,NSF_2324157,NSF,Collaborative Research: DMREF: Computationally Driven Discovery and Synthesis of 2D Materials through Selective Etching,AI-enabled catalysis,2023,7.5,0.14881,0.500,0.433333,0.0,53.1
8,9,NSF_2334969,NSF,Collaborative Research: Beyond the Single-Atom Paradigm: A Priori Design of Dual-Atom Alloy Active Sites for Efficie...,AI-enabled catalysis,2024,6.4,0.50000,0.625,0.300000,0.0,53.0
9,10,NSF_2413579,NSF,Collaborative Research: DMREF: Closed-Loop Design of Polymers with Adaptive Networks for Extreme Mechanics,Materials informatics,2024,8.6,0.00000,0.125,0.533333,0.0,52.8


Results returned: 10
Duplicate title families: 0


In [25]:
# Add explainable match terms to the final recommendation results

improved_feature_names = (
    improved_tfidf_vectorizer.get_feature_names_out()
)

final_explanation_stop_terms = {
    "research",
    "project",
    "projects",
    "study",
    "studies",
    "method",
    "methods",
    "development",
    "develop",
    "using",
    "use",
    "new",
    "work",
}


def get_final_shared_terms(
    query,
    model_row_id,
    top_k=8,
):
    """Return the strongest TF-IDF terms shared by query and grant."""

    normalized_query = normalize_model_text(query)

    query_vector = (
        improved_tfidf_vectorizer.transform(
            [normalized_query]
        )
    )

    document_vector = (
        improved_grant_tfidf_matrix[
            int(model_row_id)
        ]
    )

    shared_vector = query_vector.multiply(
        document_vector
    ).tocsr()

    if shared_vector.nnz == 0:
        return []

    ranked_features = sorted(
        zip(
            shared_vector.indices,
            shared_vector.data,
        ),
        key=lambda item: item[1],
        reverse=True,
    )

    selected_terms = []

    for feature_index, _ in ranked_features:
        term = improved_feature_names[
            feature_index
        ]

        if term in final_explanation_stop_terms:
            continue

        if term not in selected_terms:
            selected_terms.append(term)

        if len(selected_terms) == top_k:
            break

    return selected_terms


final_demo_results_df["shared_terms"] = (
    final_demo_results_df.apply(
        lambda row: get_final_shared_terms(
            query=demo_query,
            model_row_id=row["model_row_id"],
            top_k=8,
        ),
        axis=1,
    )
)

final_demo_results_df["why_it_matched"] = (
    final_demo_results_df.apply(
        lambda row: (
            f"Shared concepts: "
            f"{', '.join(row['shared_terms'])}. "
            f"Domain evidence: "
            f"{row['specific_domain_score']:.2f}; "
            f"context: {row['context_score']:.2f}; "
            f"workflow: {row['workflow_score']:.2f}."
        ),
        axis=1,
    )
)

display(
    final_demo_results_df[
        [
            "rank",
            "title",
            "source",
            "final_score_pct",
            "shared_terms",
            "why_it_matched",
        ]
    ]
)

print(
    "Results with explanations:",
    final_demo_results_df[
        "shared_terms"
    ].str.len().gt(0).sum(),
    "/",
    len(final_demo_results_df),
)

,rank,title,source,final_score_pct,shared_terms,why_it_matched
0,1,EAGER: ADAPT: Hypotheses Generation in Heterogeneous Catalysis using Causal Inference and Machine Learning,NSF,81.0,"[catalyst using, heterogeneous catalyst, heterogeneous, catalyst, discovery, machine learning, machine, learning]","Shared concepts: catalyst using, heterogeneous catalyst, heterogeneous, catalyst, discovery, machine learning, machi..."
1,2,"Collaborative Research: DMREF: Machine Learning-aided Discovery of Synthesizable, Active and Stable Heterogeneous Ca...",NSF,78.5,"[stable heterogeneous, heterogeneous catalyst, stable, catalyst, heterogeneous, loop, discovery, experimentation]","Shared concepts: stable heterogeneous, heterogeneous catalyst, stable, catalyst, heterogeneous, loop, discovery, exp..."
2,3,Conference: Artificial Intelligence for Multidisciplinary Exploration and Discovery (AIMED) in Heterogeneous Catalys...,NSF,64.5,"[heterogeneous catalyst, heterogeneous, catalyst, discovery, experimentation]","Shared concepts: heterogeneous catalyst, heterogeneous, catalyst, discovery, experimentation. Domain evidence: 1.00;..."
3,4,CAREER: Learning mechanistic models with automated experiments,NSF,61.5,"[automation experimentation, automation, closed loop, closed, loop, experimentation, learning]","Shared concepts: automation experimentation, automation, closed loop, closed, loop, experimentation, learning. Domai..."
4,5,Bimetallic Single-Site Heterogeneous Catalysts: A New Paradigm for Enhancing Reactivity in Olefin Metathesis,NSF,57.8,"[heterogeneous catalyst, heterogeneous, catalyst, experimentation, machine learning, machine, learning]","Shared concepts: heterogeneous catalyst, heterogeneous, catalyst, experimentation, machine learning, machine, learni..."
5,6,Computational Studies on Heterogeneous Astrocatalysis of Space-Abundant Transition Metals,CORDIS,56.5,"[heterogeneous catalyst, catalyst using, heterogeneous, catalyst, machine learning, machine, learning]","Shared concepts: heterogeneous catalyst, catalyst using, heterogeneous, catalyst, machine learning, machine, learnin..."
6,7,CAREER: Advancing Light-mediated Ni Catalysis using Data Science and Physical Organic Techniques,NSF,55.3,"[catalyst using, catalyst, optimization]","Shared concepts: catalyst using, catalyst, optimization. Domain evidence: 1.00; context: 0.62; workflow: 0.07."
7,8,Collaborative Research: DMREF: Computationally Driven Discovery and Synthesis of 2D Materials through Selective Etching,NSF,53.1,"[loop optimization, closed loop, closed, loop, discovery, stable, catalyst, experimentation]","Shared concepts: loop optimization, closed loop, closed, loop, discovery, stable, catalyst, experimentation. Domain ..."
8,9,Collaborative Research: Beyond the Single-Atom Paradigm: A Priori Design of Dual-Atom Alloy Active Sites for Efficie...,NSF,53.0,"[catalyst using, heterogeneous catalyst, catalyst, stable, heterogeneous, loop, experimentation, discovery]","Shared concepts: catalyst using, heterogeneous catalyst, catalyst, stable, heterogeneous, loop, experimentation, dis..."
9,10,Collaborative Research: DMREF: Closed-Loop Design of Polymers with Adaptive Networks for Extreme Mechanics,NSF,52.8,"[using automation, closed loop, loop, automation experimentation, closed, learning guided, automation, discovery]","Shared concepts: using automation, closed loop, loop, automation experimentation, closed, learning guided, automatio..."


Results with explanations: 10 / 10


In [26]:
# Test the final model with several distinct research concepts

model_test_queries = [
    {
        "query_id": "Q1",
        "query_label": "Catalyst discovery",
        "query_text": demo_query,
    },
    {
        "query_id": "Q2",
        "query_label": "Sustainable polymer materials",
        "query_text": (
            "Machine-learning-guided discovery of sustainable "
            "polymer materials with recyclable properties and "
            "reduced environmental impact."
        ),
    },
    {
        "query_id": "Q3",
        "query_label": "Autonomous reaction laboratory",
        "query_text": (
            "An autonomous robotic laboratory for closed-loop "
            "chemical reaction optimization using machine learning "
            "and automated experimentation."
        ),
    },
]

test_result_frames = []

for test_query in model_test_queries:
    query_results_df = context_aware_similarity_search(
        query=test_query["query_text"],
        top_n=5,
    ).copy()

    query_results_df["query_id"] = test_query["query_id"]
    query_results_df["query_label"] = test_query["query_label"]
    query_results_df["query_text"] = test_query["query_text"]

    query_results_df["shared_terms"] = (
        query_results_df.apply(
            lambda row: get_final_shared_terms(
                query=test_query["query_text"],
                model_row_id=row["model_row_id"],
                top_k=6,
            ),
            axis=1,
        )
    )

    test_result_frames.append(query_results_df)

model_test_results_df = pd.concat(
    test_result_frames,
    ignore_index=True,
)

display(
    model_test_results_df[
        [
            "query_id",
            "query_label",
            "rank",
            "source",
            "title",
            "primary_topic",
            "award_year",
            "final_score_pct",
            "shared_terms",
        ]
    ]
)

generalization_check_df = (
    model_test_results_df.groupby(
        ["query_id", "query_label"],
        as_index=False,
    )
    .agg(
        results_returned=("grant_key", "size"),
        unique_title_families=(
            "project_title_key",
            "nunique",
        ),
        represented_sources=("source", "nunique"),
        represented_topics=("primary_topic", "nunique"),
        explained_results=(
            "shared_terms",
            lambda values: sum(
                len(terms) > 0 for terms in values
            ),
        ),
    )
)

display(generalization_check_df)

,query_id,query_label,rank,source,title,primary_topic,award_year,final_score_pct,shared_terms
0,Q1,Catalyst discovery,1,NSF,EAGER: ADAPT: Hypotheses Generation in Heterogeneous Catalysis using Causal Inference and Machine Learning,AI-enabled catalysis,2022,81.0,"[catalyst using, heterogeneous catalyst, heterogeneous, catalyst, discovery, machine learning]"
1,Q1,Catalyst discovery,2,NSF,"Collaborative Research: DMREF: Machine Learning-aided Discovery of Synthesizable, Active and Stable Heterogeneous Ca...",AI-enabled catalysis,2022,78.5,"[stable heterogeneous, heterogeneous catalyst, stable, catalyst, heterogeneous, loop]"
2,Q1,Catalyst discovery,3,NSF,Conference: Artificial Intelligence for Multidisciplinary Exploration and Discovery (AIMED) in Heterogeneous Catalys...,AI-enabled catalysis,2024,64.5,"[heterogeneous catalyst, heterogeneous, catalyst, discovery, experimentation]"
3,Q1,Catalyst discovery,4,NSF,CAREER: Learning mechanistic models with automated experiments,Autonomous laboratories,2024,61.5,"[automation experimentation, automation, closed loop, closed, loop, experimentation]"
4,Q1,Catalyst discovery,5,NSF,Bimetallic Single-Site Heterogeneous Catalysts: A New Paradigm for Enhancing Reactivity in Olefin Metathesis,AI-enabled catalysis,2024,57.8,"[heterogeneous catalyst, heterogeneous, catalyst, experimentation, machine learning, machine]"
5,Q2,Sustainable polymer materials,1,CORDIS,Toward Desirable Metal Organic Framework Mixed Matrix Materials through Machine learning-guided Interface Design,Materials informatics,2023,72.0,"[learning guided, guided, properties, materials, discovery, machine learning]"
6,Q2,Sustainable polymer materials,2,CORDIS,Next generation toolbox for greener pharmaceuticals design & manufacturing towards reduced environmental impact,AI-enabled chemistry,2022,66.0,"[reduced environmental, environmental impact, reduced, environmental, impact]"
7,Q2,Sustainable polymer materials,3,NSF,Equipment: MRI: Track 2 Acquisition of an Automated High-Throughput System for Combinatorial Design and Development ...,AI-enabled materials,2023,60.3,"[polymer materials, polymer, discovery, guided, materials, sustainable]"
8,Q2,Sustainable polymer materials,4,NSF,Collaborative Research: Mechanisms of Catalytic Enhancement of Immobilized Lipases by Tunable Polymer Materials,AI-enabled catalysis,2024,56.6,"[polymer materials, polymer, reduced, materials, impact, properties]"
9,Q2,Sustainable polymer materials,5,NSF,Computational discovery of block polymer materials,AI-enabled materials,2025,55.8,"[polymer materials, polymer, discovery, materials, properties, learning]"


,query_id,query_label,results_returned,unique_title_families,represented_sources,represented_topics,explained_results
0,Q1,Catalyst discovery,5,5,1,2,5
1,Q2,Sustainable polymer materials,5,5,2,4,5
2,Q3,Autonomous reaction laboratory,5,5,2,4,5


### Generalization check

The final model was tested with three distinct research concepts: catalyst discovery, sustainable polymer materials, and autonomous reaction laboratories.

Each query returned five unique project families with an explanation of the shared terms. The catalyst and autonomous-laboratory results were especially coherent, while the sustainable-materials query included one broader environmental match.

These tests suggest that the model can generalize beyond the development example, although recommendation quality still depends on the wording of the query and the available grant corpus.

In [27]:
# Compare benchmark ranking performance across model versions

def summarize_benchmark_model(
    model_name,
    evaluation_df,
    rank_column,
):
    """Summarize ranking quality against the manual benchmark."""

    relevant_mask = (
        evaluation_df["manual_relevance_score"] >= 2
    )

    strong_mask = (
        evaluation_df["manual_relevance_score"] == 3
    )

    weak_mask = (
        evaluation_df["manual_relevance_score"] == 1
    )

    relevant_ranks = pd.to_numeric(
        evaluation_df.loc[
            relevant_mask,
            rank_column,
        ],
        errors="coerce",
    ).dropna()

    strong_ranks = pd.to_numeric(
        evaluation_df.loc[
            strong_mask,
            rank_column,
        ],
        errors="coerce",
    ).dropna()

    weak_ranks = pd.to_numeric(
        evaluation_df.loc[
            weak_mask,
            rank_column,
        ],
        errors="coerce",
    ).dropna()

    weak_rank = (
        weak_ranks.min()
        if not weak_ranks.empty
        else np.nan
    )

    relevant_above_weak = (
        (relevant_ranks < weak_rank).sum()
        if not pd.isna(weak_rank)
        else np.nan
    )

    return {
        "model": model_name,
        "relevant_in_top_10": (
            relevant_ranks <= 10
        ).sum(),
        "relevant_in_top_25": (
            relevant_ranks <= 25
        ).sum(),
        "relevant_in_top_50": (
            relevant_ranks <= 50
        ).sum(),
        "median_relevant_rank": round(
            relevant_ranks.median(),
            1,
        ),
        "best_strong_match_rank": (
            strong_ranks.min()
            if not strong_ranks.empty
            else np.nan
        ),
        "weak_match_rank": weak_rank,
        "relevant_above_weak_match": (
            relevant_above_weak
        ),
    }


model_comparison_df = pd.DataFrame(
    [
        summarize_benchmark_model(
            model_name="Initial TF-IDF",
            evaluation_df=tfidf_evaluation_df,
            rank_column="tfidf_rank",
        ),
        summarize_benchmark_model(
            model_name="Normalized TF-IDF",
            evaluation_df=improved_evaluation_df,
            rank_column="improved_tfidf_rank",
        ),
        summarize_benchmark_model(
            model_name="Concept-aware hybrid",
            evaluation_df=hybrid_evaluation_df,
            rank_column="hybrid_rank",
        ),
        summarize_benchmark_model(
            model_name="Field-aware hybrid",
            evaluation_df=field_aware_evaluation_df,
            rank_column="field_aware_rank",
        ),
        summarize_benchmark_model(
            model_name="Context-aware hybrid",
            evaluation_df=context_evaluation_df,
            rank_column="context_aware_rank",
        ),
    ]
)

# Convert rank/count columns to nullable integers
integer_columns = [
    "relevant_in_top_10",
    "relevant_in_top_25",
    "relevant_in_top_50",
    "best_strong_match_rank",
    "weak_match_rank",
    "relevant_above_weak_match",
]

model_comparison_df[integer_columns] = (
    model_comparison_df[integer_columns]
    .astype("Int64")
)

display(model_comparison_df)

print(
    "Benchmark denominator: "
    "7 relevant/strong projects and 1 weak project."
)

print(
    "Higher is better for top-k counts, weak-match rank, "
    "and relevant projects above the weak match."
)

print(
    "Lower is better for median relevant rank "
    "and best strong-match rank."
)

,model,relevant_in_top_10,relevant_in_top_25,relevant_in_top_50,median_relevant_rank,best_strong_match_rank,weak_match_rank,relevant_above_weak_match
0,Initial TF-IDF,0,1,1,76.0,54,138,4
1,Normalized TF-IDF,0,1,2,93.0,41,54,2
2,Concept-aware hybrid,0,1,5,42.0,26,27,2
3,Field-aware hybrid,0,1,5,32.0,15,87,5
4,Context-aware hybrid,0,3,5,39.0,14,212,7


Benchmark denominator: 7 relevant/strong projects and 1 weak project.
Higher is better for top-k counts, weak-match rank, and relevant projects above the weak match.
Lower is better for median relevant rank and best strong-match rank.


### Model comparison

The context-aware hybrid was selected as the final ranking model.

It placed three of seven relevant benchmark projects in the top 25 and five in the top 50. Its strongest manually reviewed match ranked 14th, while the weak drug-discovery comparison fell to rank 212.

Although the field-aware model had a slightly lower median relevant rank, the context-aware model provided the clearest overall separation: all seven relevant projects ranked above the weak comparison.

No benchmark project reached the top ten, so the model should be presented as a research-discovery and prioritization tool rather than a definitive relevance classifier.

In [28]:
# Export the final model package and supporting evaluation files

import json
import joblib
from scipy.sparse import save_npz

MODEL_DATA_DIR = Path(
    "../Data/Processed_Data/Model"
)

MODEL_ASSET_DIR = Path(
    "../Models"
)

MODEL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_ASSET_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def prepare_dataframe_for_csv(dataframe):
    """Convert list and dictionary values into JSON strings."""

    export_df = dataframe.copy()

    for column in export_df.columns:
        contains_complex_values = export_df[
            column
        ].apply(
            lambda value: isinstance(
                value,
                (list, dict, tuple, set),
            )
        ).any()

        if contains_complex_values:
            export_df[column] = export_df[
                column
            ].apply(
                lambda value: json.dumps(
                    list(value)
                    if isinstance(value, set)
                    else value,
                    ensure_ascii=False,
                )
                if isinstance(
                    value,
                    (list, dict, tuple, set),
                )
                else value
            )

    return export_df


# 1. Search catalogue aligned with the TF-IDF matrix
grant_search_catalog_export_df = (
    prepare_dataframe_for_csv(
        model_document_df
    )
)

grant_search_catalog_export_df.to_csv(
    MODEL_DATA_DIR
    / "grant_search_catalog.csv",
    index=False,
)


# 2. Demonstration outputs
prepare_dataframe_for_csv(
    demo_keyword_results_df
).to_csv(
    MODEL_DATA_DIR
    / "demo_keyword_baseline_results.csv",
    index=False,
)

prepare_dataframe_for_csv(
    improved_demo_results_df
).to_csv(
    MODEL_DATA_DIR
    / "demo_tfidf_similarity_results.csv",
    index=False,
)

prepare_dataframe_for_csv(
    final_demo_results_df
).to_csv(
    MODEL_DATA_DIR
    / "demo_final_context_aware_results.csv",
    index=False,
)


# 3. Benchmark and model-comparison outputs
prepare_dataframe_for_csv(
    context_evaluation_df
).to_csv(
    MODEL_DATA_DIR
    / "similarity_evaluation_results.csv",
    index=False,
)

model_comparison_df.to_csv(
    MODEL_DATA_DIR
    / "model_comparison.csv",
    index=False,
)


# 4. Generalization-test inputs and results
model_test_queries_df = pd.DataFrame(
    model_test_queries
)

model_test_queries_df.to_csv(
    MODEL_DATA_DIR
    / "model_test_queries.csv",
    index=False,
)

prepare_dataframe_for_csv(
    model_test_results_df
).to_csv(
    MODEL_DATA_DIR
    / "model_test_results.csv",
    index=False,
)

generalization_check_df.to_csv(
    MODEL_DATA_DIR
    / "generalization_check.csv",
    index=False,
)


# 5. Model metadata
model_metadata_df = pd.DataFrame(
    {
        "parameter": [
            "model_name",
            "model_type",
            "catalogue_documents",
            "source_grant_records",
            "vocabulary_size",
            "matrix_rows",
            "matrix_columns",
            "matrix_nonzero_values",
            "title_weight",
            "ngram_range",
            "minimum_document_frequency",
            "maximum_document_frequency",
            "tfidf_weight",
            "specific_domain_weight",
            "context_weight",
            "workflow_weight",
            "off_domain_penalty_weight",
            "benchmark_projects",
            "relevant_benchmark_projects",
            "selected_model",
            "exported_at",
        ],
        "value": [
            "GrantScopeAI context-aware hybrid",
            (
                "Unsupervised TF-IDF retrieval with "
                "expert-defined scientific ranking rules"
            ),
            len(model_document_df),
            len(model_catalog_df),
            len(
                improved_tfidf_vectorizer.vocabulary_
            ),
            improved_grant_tfidf_matrix.shape[0],
            improved_grant_tfidf_matrix.shape[1],
            improved_grant_tfidf_matrix.nnz,
            3,
            "(1, 2)",
            2,
            0.95,
            0.58,
            0.17,
            0.12,
            0.13,
            0.10,
            len(context_evaluation_df),
            (
                context_evaluation_df[
                    "manual_relevance_score"
                ] >= 2
            ).sum(),
            True,
            pd.Timestamp.now().isoformat(),
        ],
    }
)

model_metadata_df.to_csv(
    MODEL_DATA_DIR
    / "model_metadata.csv",
    index=False,
)


# 6. Configuration used by the context-aware ranking
final_model_configuration = {
    "model_name": (
        "GrantScopeAI context-aware hybrid"
    ),
    "title_weight": 3,
    "tfidf_parameters": {
        "stop_words": "english",
        "ngram_range": [1, 2],
        "min_df": 2,
        "max_df": 0.95,
        "sublinear_tf": True,
    },
    "ranking_weights": {
        "tfidf": 0.58,
        "specific_domain": 0.17,
        "context": 0.12,
        "workflow": 0.13,
        "off_domain_penalty": 0.10,
    },
    "domain_concept_lexicon": (
        DOMAIN_CONCEPT_LEXICON
    ),
    "workflow_concept_lexicon": (
        WORKFLOW_CONCEPT_LEXICON
    ),
    "context_concept_lexicon": (
        CONTEXT_CONCEPT_LEXICON
    ),
    "domain_context_expansion": (
        DOMAIN_CONTEXT_EXPANSION
    ),
    "application_lexicon": (
        APPLICATION_LEXICON
    ),
    "application_penalties": (
        APPLICATION_PENALTIES
    ),
}

with open(
    MODEL_DATA_DIR
    / "model_configuration.json",
    "w",
    encoding="utf-8",
) as configuration_file:
    json.dump(
        final_model_configuration,
        configuration_file,
        indent=2,
        ensure_ascii=False,
    )


# 7. Fitted machine-learning assets
joblib.dump(
    improved_tfidf_vectorizer,
    MODEL_ASSET_DIR
    / "tfidf_vectorizer.joblib",
)

save_npz(
    MODEL_ASSET_DIR
    / "grant_tfidf_matrix.npz",
    improved_grant_tfidf_matrix,
)


# Summarize the export
exported_files = sorted(
    list(MODEL_DATA_DIR.glob("*"))
    + list(MODEL_ASSET_DIR.glob("*"))
)

export_summary_df = pd.DataFrame(
    {
        "file": [
            file_path.name
            for file_path in exported_files
        ],
        "folder": [
            file_path.parent.name
            for file_path in exported_files
        ],
        "size_kb": [
            round(
                file_path.stat().st_size / 1024,
                1,
            )
            for file_path in exported_files
        ],
    }
)

display(export_summary_df)

print(
    "Exported files:",
    len(export_summary_df),
)

print(
    "Catalogue rows:",
    len(grant_search_catalog_export_df),
)

print(
    "TF-IDF matrix shape:",
    improved_grant_tfidf_matrix.shape,
)

,file,folder,size_kb
0,demo_final_context_aware_results.csv,Model,164.0
1,demo_keyword_baseline_results.csv,Model,109.4
2,demo_tfidf_similarity_results.csv,Model,144.7
3,generalization_check.csv,Model,0.2
4,grant_search_catalog.csv,Model,44294.9
5,model_comparison.csv,Model,0.3
6,model_configuration.json,Model,2.8
7,model_metadata.csv,Model,0.6
8,model_test_queries.csv,Model,0.5
9,model_test_results.csv,Model,242.4


Exported files: 13
Catalogue rows: 2943
TF-IDF matrix shape: (2943, 103714)


In [29]:
# Reload and validate the exported model package

from scipy.sparse import load_npz

# Reload core model assets
reloaded_catalog_df = pd.read_csv(
    MODEL_DATA_DIR / "grant_search_catalog.csv"
)

reloaded_vectorizer = joblib.load(
    MODEL_ASSET_DIR / "tfidf_vectorizer.joblib"
)

reloaded_tfidf_matrix = load_npz(
    MODEL_ASSET_DIR / "grant_tfidf_matrix.npz"
)

# Reload supporting files
reloaded_configuration_path = (
    MODEL_DATA_DIR / "model_configuration.json"
)

with open(
    reloaded_configuration_path,
    "r",
    encoding="utf-8",
) as configuration_file:
    reloaded_configuration = json.load(
        configuration_file
    )

reloaded_metadata_df = pd.read_csv(
    MODEL_DATA_DIR / "model_metadata.csv"
)

reloaded_evaluation_df = pd.read_csv(
    MODEL_DATA_DIR
    / "similarity_evaluation_results.csv"
)

reloaded_test_results_df = pd.read_csv(
    MODEL_DATA_DIR / "model_test_results.csv"
)

# Validate catalogue and matrix alignment
validation_results = {
    "Catalogue rows equal matrix rows": (
        len(reloaded_catalog_df)
        == reloaded_tfidf_matrix.shape[0]
    ),
    "Matrix columns equal vocabulary size": (
        reloaded_tfidf_matrix.shape[1]
        == len(reloaded_vectorizer.vocabulary_)
    ),
    "Model row IDs are unique": (
        reloaded_catalog_df[
            "model_row_id"
        ].duplicated().sum()
        == 0
    ),
    "Model row IDs align with row order": (
        reloaded_catalog_df[
            "model_row_id"
        ].tolist()
        == list(range(len(reloaded_catalog_df)))
    ),
    "No missing search text": (
        reloaded_catalog_df[
            "weighted_search_text"
        ].isna().sum()
        == 0
    ),
    "Configuration contains ranking weights": (
        "ranking_weights"
        in reloaded_configuration
    ),
    "Evaluation contains eight benchmark rows": (
        len(reloaded_evaluation_df) == 8
    ),
    "Generalization results contain fifteen rows": (
        len(reloaded_test_results_df) == 15
    ),
}

validation_df = pd.DataFrame(
    {
        "validation_check": (
            validation_results.keys()
        ),
        "passed": validation_results.values(),
    }
)

display(validation_df)

# Confirm that the reloaded model can score the demo query
reloaded_query_vector = (
    reloaded_vectorizer.transform(
        [normalize_model_text(demo_query)]
    )
)

reloaded_similarity_scores = cosine_similarity(
    reloaded_query_vector,
    reloaded_tfidf_matrix,
).ravel()

reloaded_top_index = (
    reloaded_similarity_scores.argmax()
)

print(
    "Validation checks passed:",
    validation_df["passed"].sum(),
    "/",
    len(validation_df),
)

print(
    "Reloaded query features:",
    reloaded_query_vector.nnz,
)

print(
    "Highest raw TF-IDF match:",
    reloaded_catalog_df.loc[
        reloaded_top_index,
        "title",
    ],
)

print(
    "Highest raw similarity (%):",
    round(
        reloaded_similarity_scores[
            reloaded_top_index
        ]
        * 100,
        1,
    ),
)

,validation_check,passed
0,Catalogue rows equal matrix rows,True
1,Matrix columns equal vocabulary size,True
2,Model row IDs are unique,True
3,Model row IDs align with row order,True
4,No missing search text,True
5,Configuration contains ranking weights,True
6,Evaluation contains eight benchmark rows,True
7,Generalization results contain fifteen rows,True


Validation checks passed: 8 / 8
Reloaded query features: 24
Highest raw TF-IDF match: EAGER: ADAPT: Hypotheses Generation in Heterogeneous Catalysis using Causal Inference and Machine Learning
Highest raw similarity (%): 11.2


## Conclusion

Notebook 07 developed and validated the GrantScopeAI similarity-search model.

The final system combines:
- normalized, title-weighted TF-IDF;
- cosine-similarity retrieval;
- scientific-domain and workflow context;
- title-family diversification;
- explainable shared-term recommendations.

The context-aware hybrid was selected because it provided the clearest separation between relevant chemistry projects and the weak off-domain comparison. It also returned coherent results across catalyst discovery, sustainable materials, and autonomous-laboratory test queries.

The model is an unsupervised research-discovery tool, not a funding-success predictor. Its recommendations depend on the available grant corpus, query wording, and expert-defined concept rules.

All catalogue, evaluation, configuration, vectorizer, and matrix files were exported and independently reloaded successfully for use in Streamlit.